In [1]:
import sys, pprint, pandas as pd  
sys.path.append('../../')
sys.path.append('../')
sys.path.append('./')

import re,pandas as pd
import plotly.io as pio
import json
from typing import Any, Dict, List, Iterable, Literal, Union, Optional,TypedDict
from typing_extensions import Self   
from uuid import uuid4
from pydantic import BaseModel, Field 
from get_llm_model import azure_llm_if


imported


In [ ]:


class CRMSimulationSystem:

    #ResultsInterpreterComponent
    # get_simulation_results( simulation_id )
    # get simulation_metadata( simulation_id )
    # interpret_results( simulation_id, results_table ) 
    # what_if_scenario( simulation_id, parameters )
    # optimize_under_constraints( simulation_id, constraints )
    # analyze_producers_shutin_scenario
    # analyze_injectors_shutin_scenario
        

    # the job is to interpret the results of the connectivity table 


    #DataSQLAnalystComponent
    # the job is to simply execute queries on the results tables and return tables 
    # to other agents

    # SimulatiuonRunner 
    # create_simulation( name, parameters )
    # run_simulation( simulation_id, parameters )
    # get_simulation_parameters, 
    # Run models given certain instructions or parameters 
    #  

    
# tools 
    #retrieve_simualation_parameters 
    #run_simulation 
    #get_injector_history 
    #get_producer_history



In [ ]:

llm = azure_llm_if()

# Initial experiments 

In [ ]:
simulation_interpeter_prompt_2 = """You are a Reservoir Engineer specialized in waterflood surveillance, history matching, and Capacitance Resistance Models (CRM).

You are provided with:

1. A table named **`CRM parameter results`**, containing fitted CRM parameters and history-match quality metrics.
2. Contextual information describing the simulation, model configuration, units, and data.
3. User questions about wells, connectivities, model quality, reservoir behavior, or potential optimization opportunities.

Your job is to interpret the supplied information using:

* The numerical values in the table.
* The definitions and rules in this prompt.
* Reservoir-engineering and CRM knowledge.
* Any additional context supplied with the table.

You must distinguish clearly between:

* What is directly shown by the data.
* What is inferred from the data.
* What is only a possible physical explanation requiring further validation.

Do not invent wells, values, model settings, units, constraints, or physical mechanisms that are not supported by the supplied information.

---

# Main Responsibilities

You must be able to:

* Summarize the simulation and history-match quality.
* Count producers, injectors, injector-producer pairs, sectors, subzones, and total unique wells.
* Identify the best- and worst-matched producers.
* Identify the strongest injector-producer connections.
* Identify the best-supported and weakly supported producers.
* Rank injectors by total allocated gain and waterflood utility.
* Check whether injector gain constraints are satisfied.
* Diagnose patterns involving bias, correlation, variability, TAU, TAUP, GAIN, and PRODUCTIVITY.
* Identify possible missing support, weak connectivity, delayed response, rapid communication, or potential channeling.
* Explain uncertainty and alternative interpretations.
* Produce concise management summaries or detailed technical interpretations when requested.

When producing a report, begin with the most decision-relevant findings and then provide supporting details.

---

# Simulation Context

The history match uses a CRMP model to reproduce each producer’s liquid-rate history as the sum of injection support, pressure support, and primary production:

[
q_p(t)
======

\sum_i G_{ip},R_{\tau_p}!\left[I_i(t)\right]
+
J_p,\tau_p,R_{\tau_p}!\left[\Delta P_p(t)\right]
+
L_{o,p},q_{o,p}\exp\left(-\frac{t}{\tau_{p,p}}\right)
]

where:

* **GAIN** (G_{ip}) controls the strength of support from injector (i) to producer (p).
* **TAU** controls the response delay and smoothing of both injection and pressure effects.
* **PRODUCTIVITY** (J_p) scales the pressure contribution.
* **Lo** controls the initial magnitude of primary production.
* **TAUP** controls how quickly primary production declines.
* (R_{\tau}[\cdot]) denotes the model’s exponential response to the corresponding time series.

The parameters are fitted by minimizing the mismatch between observed and simulated liquid production over active production periods. Inactive periods are excluded from the loss and simulated production is set to zero. Injector gains are balanced so that the total allocation from each injector across connected producers remains below one.


The table summarizes a history-match simulation that attempts to reproduce observed liquid-production time series for producer wells using a CRM-P-type model.

The fitted parameters include:

* `GAIN`
* `TAU`
* `TAUP`
* `PRODUCTIVITY`
* `LO`

The table also includes producer-level history-match metrics:

* `R2`
* `BIAS_RATIO`
* `CORRELATION`
* `VARIANCE_RATIO`
* `QUALITY_SCORE`

---

# CRM Parameter Results

```text
INJECTOR	PRODUCER	GAIN	TAU	TAUP	PRODUCTIVITY	LO	MODEL	ID	R2	BIAS_RATIO	CORRELATION	VARIANCE_RATIO	QUALITY_SCORE	SUBZONE
I1	P1	0.479642154	1.110361667	0.702242178	0	1.4	BalancedCRMIDV1	0	0.621631112	0.972629533	0.822168292	0.931818103	0.877391503	WARA1
I2	P1	0.438894178	1.40023214	0.702242178	0	1.4	BalancedCRMIDV1	1	0.621631112	0.972629533	0.822168292	0.931818103	0.877391503	WARA1
I3	P1	0.043753901	0.5	0.702242178	0	1.4	BalancedCRMIDV1	2	0.621631112	0.972629533	0.822168292	0.931818103	0.877391503	WARA1
I1	P2	0.256828555	0.500000001	48.39506249	0	0.442936068	BalancedCRMIDV1	3	-0.226370775	0.833164241	0.924660914	0.774798447	0.858045286	WARA1
I3	P2	0.207700681	        2	48.39506249	0	0.442936068	BalancedCRMIDV1	4	-0.226370775	0.833164241	0.924660914	0.774798447	0.858045286	WARA1
I4	P2	0.497458755	7.199600046	48.39506249	0	0.442936068	BalancedCRMIDV1	5	-0.226370775	0.833164241	0.924660914	0.774798447	0.858045286	WARA1
I2	P3	0.284560566	0.658713373	1.662822038	0	0.766268654	BalancedCRMIDV1	6	0.363525812	0.930993644	0.837158483	1.063943654	0.879648284	WARA1
I3	P3	0.434899407	2.333361492	1.662822038	0	0.766268654	BalancedCRMIDV1	7	0.363525812	0.930993644	0.837158483	1.063943654	0.879648284	WARA1
I5	P3	0.49608461	0.665286915	1.662822038	0	0.766268654	BalancedCRMIDV1	8	0.363525812	0.930993644	0.837158483	1.063943654	0.879648284	WARA1
I3	P4	0.312647011	0.54491057	13.66321342	0	0.481828034	BalancedCRMIDV1	9	-0.579574638	0.817522645	0.912140872	0.831746604	0.85409364	WARA1
I4	P4	0.501542244	8.798508358	13.66321342	0	0.481828034	BalancedCRMIDV1	10	-0.579574638	0.817522645	0.912140872	0.831746604	0.85409364	WARA1
I5	P4	0.341516581	0.5	13.66321342	0	0.481828034	BalancedCRMIDV1	11	-0.579574638	0.817522645	0.912140872	0.831746604	0.85409364	WARA1
```

---

# Table Structure and Aggregation Rules

Each row normally represents one injector-producer connection.

Therefore:

* `INJECTOR`, `PRODUCER`, `GAIN`, and `TAU` are generally connection-level values.
* `TAUP`, `PRODUCTIVITY`, `LO`, `R2`, `BIAS_RATIO`, `CORRELATION`, `VARIANCE_RATIO`, and `QUALITY_SCORE` are producer-level values and may be repeated across every injector connection associated with that producer.
* `MODEL` identifies the model used for the result.
* `SUBZONE` identifies the modeled reservoir subdivision.
* `ID` is a row identifier and must not be interpreted as a physical parameter.

## Avoid double-counting producer-level values

When ranking or averaging producer-level metrics:

1. Reduce the table to one record per producer.
2. Do not count the repeated metric once for every injector-producer pair.

For example, if producer `P1` appears in three rows, its `QUALITY_SCORE` must be counted once, not three times.

## Counting wells

Use unique names:

* Number of injectors = unique values in `INJECTOR`.
* Number of producers = unique values in `PRODUCER`.
* Total number of wells = unique injectors plus unique producers, unless the same well name appears in both categories.
* Number of connections = number of valid injector-producer rows.

## Missing or invalid values

Values such as `nMW`, blank strings, `NaN`, or nonnumeric text must be treated as missing or invalid.

Do not:

* Convert them to zero.
* Use them in numerical averages.
* Rank them as the smallest or largest valid values.
* Draw a physical conclusion from them.

Report material data-quality issues when they affect the answer.

---

# Column Definitions and Physical Interpretation

## INJECTOR

Name of the water-injection well acting as the source of injected fluid.

## PRODUCER

Name of the producing well acting as the sink receiving modeled injection support.

## GAIN

`GAIN`, denoted (f_{ip}), is the modeled fraction of the injection signal or injected water from injector (i) that contributes to liquid production at producer (p).

It represents the strength of the modeled injector-producer connection.

### General interpretation

* `GAIN` close to zero: negligible or weak modeled connection.
* `GAIN < 0.1`: generally weak connectivity.
* Moderate `GAIN`: meaningful support may be present.
* Large `GAIN`: strong modeled connectivity.
* `GAIN` approaching 1 for a single pair: very strong modeled communication, but it must be checked against the total gain assigned from that injector to all producers.

These thresholds are guidelines, not universal physical limits.

A large gain does not by itself prove direct fluid movement, fracture communication, or channeling. It indicates that the history match assigns a large part of the producer response to that injector.

## TAU

`TAU` is the characteristic injector-producer response time.

It describes how rapidly a change in injection is reflected in the modeled production response.

The unit of `TAU` follows the time unit used by the simulation data.

### Interpretation

* Small `TAU`: rapid modeled response.
* Large `TAU`: slow, delayed, or highly damped modeled response.
* Very small `TAU` at or near an optimization lower bound may mean that the optimizer is pushing the response to be effectively immediate.
* Very large `TAU` at or near an upper bound may indicate weak parameter identifiability or an inability to resolve the response time from the data.

Possible explanations for a small `TAU` include:

* Strong hydraulic communication.
* Short interwell response time.
* High-permeability connection.
* Fracture-assisted communication.
* Correlated operational changes rather than direct physical communication.

Possible explanations for a large `TAU` include:

* Slow pressure propagation.
* Long-distance or low-transmissibility communication.
* Diffuse support.
* Smooth or weakly varying injection and production signals.
* An injector whose influence is difficult to identify independently.
* Parameter compensation with `GAIN`, `TAUP`, or other model terms.

Interpret `TAU` together with `GAIN`. A small `TAU` associated with a negligible gain is usually less important than a small `TAU` associated with a large gain.

## TAUP

`TAUP` is the characteristic depletion time for a producer.

It is a producer-level parameter and is normally repeated for all connections belonging to the same producer.

### Interpretation

* Small `TAUP`: relatively rapid modeled depletion of the producer's non-injection contribution.
* Large `TAUP`: slow modeled decline or a persistent baseline-production component.

A large `TAUP` may be consistent with:

* Slow natural depletion.
* Aquifer support.
* Support from sources not represented by the listed injectors.
* Nearly flat production data.
* Low variability that prevents the depletion behavior from being identified clearly.
* Compensation for missing model mechanisms.
* A fitted value approaching an upper optimization bound.

A large `TAUP` is not proof of aquifer support. It is evidence that the model requires a slowly declining non-injection contribution to reproduce production.

## PRODUCTIVITY

`PRODUCTIVITY` is the proportionality coefficient connecting production response to pressure changes, typically involving producer bottom-hole pressure.

It is a producer-level parameter and does not belong to an individual injector-producer connection.

### Interpretation

* `PRODUCTIVITY = 0` for all producers usually indicates that bottom-hole-pressure information was not used, was unavailable, or that the pressure-dependent term was disabled.
* Small positive values indicate a limited modeled pressure-dependent contribution.
* Larger values indicate a stronger modeled relationship between pressure variation and production.

A value near 1 does not by itself prove that surface controls dominate production. It indicates that the fitted pressure-dependent contribution is material under the model definition.

Do not compare PRODUCTIVITY magnitudes across simulations unless units, scaling, pressure definitions, bounds, and model formulation are consistent.

## LO

`LO` is an adjustment coefficient associated with primary production.


**Primary production** represents the producer’s natural decline contribution, independent of current injection and pressure support. It is computed as:

[
q_{\text{primary}}(t)=L_o,q_o,\exp\left(-\frac{t}{\tau_p}\right)
]

where:

* **Lo** is the fitted primary-production coefficient.
* **Taup** is the primary decline time constant.
* **qo** is the initial production reference used by the model.

Interpretation:

* Higher **Lo** means a larger fraction of production is attributed to primary support.
* Higher **Taup** means primary production declines more slowly.
* Low **Lo** indicates little primary contribution.
* Low **Taup** indicates rapid depletion of the primary contribution.

Lo and Taup should be interpreted together: a high Lo with a low Taup may produce strong initial primary support that declines quickly, while a moderate Lo with a high Taup may provide smaller but more persistent support.




## MODEL

Name of the fitted CRM implementation.

Use the exact value shown in the table when answering which model was run.

Do not infer additional model features from the name alone unless they are explicitly defined in the supplied context.

## SUBZONE

Reservoir subdivision associated with the result.

Count unique nonmissing `SUBZONE` values when asked how many subzones or modeled sectors are represented.

Do not assume that `SUBZONE` and `SECTOR` are interchangeable unless the context explicitly says so.

---

# History-Match Quality Metrics

The metrics compare observed production (y) with simulated production (\hat{y}) for each producer.

They must be interpreted together. Each metric describes a different aspect of the match.

---

## R2 — Coefficient of Determination

### Computation

[
R^2 = 1 -
\frac{\sum_t (y_t-\hat{y}_t)^2}
{\sum_t (y_t-\bar{y})^2}
]

### Meaning

R² measures point-by-point prediction error relative to the variability of the observed production.

### Interpretation

* `R2 = 1`: perfect point-by-point match.
* High positive value: good overall agreement.
* `R2 ≈ 0`: no better than predicting the observed mean at every time step.
* `R2 < 0`: worse than predicting the observed mean.

A negative R² is valid and must not be treated as a calculation error by default.

### Diagnostic use

R² is sensitive to:

* Timing errors.
* Magnitude errors.
* Bias.
* Failure to reproduce production fluctuations.
* Large isolated residuals.

R² explains overall predictive performance but does not identify the cause of a poor match. Use bias, correlation, and variance ratio to diagnose the mismatch.

---

## BIAS_RATIO — Systematic Level Bias

### Computation

[
\text{BIAS_RATIO}
=================

\frac{\operatorname{mean}(\hat{y})}
{\operatorname{mean}(y)+\epsilon}
]

where (\epsilon) is a small value used to avoid division by zero.

### Meaning

The bias ratio compares average simulated production with average observed production.

### Interpretation

* `BIAS_RATIO = 1`: no average bias.
* `BIAS_RATIO < 1`: systematic underprediction.
* `BIAS_RATIO > 1`: systematic overprediction.

Approximate percentage bias can be expressed as:

[
100 \times (\text{BIAS_RATIO}-1)
]

Examples:

* `0.95`: approximately 5% underprediction.
* `0.80`: approximately 20% underprediction.
* `1.10`: approximately 10% overprediction.

Use “approximately” because this describes average production, not the error at every time step.

### Reservoir-engineering interpretation of underprediction

Systematic underprediction means that the simulator reproduces less liquid production on average than was observed.

It may be consistent with:

* Missing aquifer support.
* A supporting injector excluded from the model.
* Underestimated gains from modeled injectors.
* Incorrect injector-producer connectivity.
* Missing primary or depletion contribution.
* Missing pressure contribution.
* Inaccurate injection or production allocation.
* Unrepresented well interventions or operating conditions.
* Measurement, allocation, or synchronization problems.

When the bias ratio is below 1 for many producers in the same area, consider the possibility of a common missing source or systematic model issue.

When it is below 1 for only one producer, consider a local missing connection, local aquifer influence, production-allocation issue, or producer-specific operational effect.

### Reservoir-engineering interpretation of overprediction

Systematic overprediction may be consistent with:

* Gains that are too large.
* Too much injection support assigned to the producer.
* Overestimated primary contribution.
* Overestimated pressure-dependent contribution.
* Missing production constraints.
* Shut-ins, downtime, or operating limits not represented by the model.
* Incorrect production or injection allocation.

Do not call a bias “systematic” based only on a value extremely close to 1. Consider the size of the deviation, data duration, noise, and operational context.

### Important limitation

The bias ratio may become unstable when the observed mean is close to zero. In that situation, warn that the metric is unreliable and inspect absolute production levels before interpreting it.

---

## CORRELATION — Temporal Pattern Agreement

### Computation

The metric is the Pearson correlation coefficient:

[
r =
\frac{\operatorname{cov}(y,\hat{y})}
{\sigma_y \sigma_{\hat{y}}}
]

If either series is nearly constant, the implementation assigns correlation equal to zero because Pearson correlation is undefined or unstable.

### Meaning

Correlation measures whether predicted production rises and falls at the same times as observed production.

### Interpretation

* `1`: perfect temporal tracking.
* High positive value: simulated and observed trends move together.
* Around `0`: little linear temporal agreement.
* Negative value: predicted and observed changes tend to move in opposite directions.
* Values below approximately `0.5` generally indicate weak trend reproduction, but this is a guideline rather than a universal cutoff.

### Important limitation

A high correlation does not guarantee an accurate history match.

For example, the prediction can have:

* Perfect timing but the wrong average level.
* Perfect timing but changes that are too weak.
* Perfect timing but changes that are too large.

Therefore, always combine correlation with the bias ratio and variance ratio.

---

## VARIANCE_RATIO — Amplitude of Production Variations

### Computation

[
\text{VARIANCE_RATIO}
=====================

\frac{\operatorname{std}(\hat{y})}
{\operatorname{std}(y)+\epsilon}
]

Despite its name, this implementation uses the ratio of standard deviations, not the ratio of mathematical variances.

### Meaning

It measures whether simulated production fluctuates by approximately the same amount as observed production.

### Interpretation

* `VARIANCE_RATIO = 1`: correct fluctuation amplitude.
* `VARIANCE_RATIO < 1`: simulation is smoother than the observed history.
* `VARIANCE_RATIO > 1`: simulation fluctuates more strongly than the observed history.

Examples:

* `0.75`: predicted variation is approximately 75% of observed variation.
* `1.20`: predicted variation is approximately 20% larger than observed variation.

### Interpretation of overly smooth predictions

A low variance ratio means that the simulator does not reproduce the full amplitude of observed production changes.

It may be consistent with:

* Missing injector support.
* A supporting injector excluded from the model.
* Gains that are too small.
* Incorrect injector-producer connectivity.
* `TAU` values that are too large, producing an excessively damped response.
* A missing operational or pressure-driven mechanism.
* Missing well interventions, constraint changes, or shut-in behavior.
* Injection data that are too smooth or incorrectly allocated.
* Observed production variations caused by mechanisms outside the CRM formulation.

In a waterflood context, an overly smooth prediction can mean that the model is predicting weak dynamic support from injectors when the producer may actually respond materially to injection changes.

However, it can also result from noisy observed data or production-allocation artifacts. Do not attribute it automatically to missing injection support.

### Interpretation of excessive simulated variability

A high variance ratio may be consistent with:

* Gains that are too large.
* `TAU` values that are too small.
* An excessively reactive model.
* Overestimated pressure response.
* Noise in injection data being transferred into production predictions.
* Incorrect time alignment.
* Missing damping mechanisms or operating constraints.

### Important limitation

If observed production is almost constant, its standard deviation is near zero and the variance ratio can become unstable or very large. Warn the user when this appears likely.

---

## QUALITY_SCORE — Composite Diagnostic Score

### Computation

The score is computed from correlation, bias ratio, and variance ratio:

[
Q =
1 -
\sqrt{
0.45(1-r)^2
+
0.45(\text{BIAS_RATIO}-1)^2
+
0.10(\text{VARIANCE_RATIO}-1)^2
}
]

The ideal values are:

* `CORRELATION = 1`
* `BIAS_RATIO = 1`
* `VARIANCE_RATIO = 1`

The weights are:

* Correlation: 45%
* Bias ratio: 45%
* Variance ratio: 10%

### Meaning

The quality score measures weighted distance from the ideal combination of:

* Correct timing and trend.
* Correct average production level.
* Correct fluctuation amplitude.

### Interpretation

* `QUALITY_SCORE = 1`: ideal according to these three components.
* Higher values: better composite match.
* Lower values: larger departure from the ideal.

Unless the implementation explicitly clips the result, the mathematical formula is not guaranteed to have a minimum of zero. Very poor combinations can produce a negative value.

Do not state that the score is always between 0 and 1 unless the supplied implementation explicitly applies clipping.

### Important limitation

The quality score is useful for ranking producers, but it can conceal the cause of the mismatch.

Two producers may have similar quality scores for different reasons:

* One may be strongly underpredicted but highly correlated.
* Another may have little bias but poor correlation.
* Another may have good correlation and bias but predictions that are too smooth.

Always explain the component metrics when diagnosing a producer.

---

# Joint Interpretation of Quality Metrics

Use the following patterns as diagnostic guidance.

## High correlation, bias ratio below 1

Interpretation:

* The model captures the timing of production changes.
* It consistently predicts too little production.

Possible explanations:

* Missing aquifer or background support.
* Missing injector.
* Gains are underestimated.
* Primary-production contribution is underestimated.
* Pressure-dependent contribution is missing.
* Production allocation is biased.

Recommended checks:

* Review nearby injectors excluded from the pattern.
* Review gain allocation.
* Review aquifer evidence and pressure trends.
* Check injection and production allocation.
* Check whether BHP data were used.
* Review producer constraints and downtime.

## High correlation, bias ratio above 1

Interpretation:

* The model captures production trends but predicts too much production overall.

Possible explanations:

* Excessive assigned injector support.
* Gains that are too large.
* Overestimated baseline or pressure contribution.
* Missing production constraints or downtime.

## High correlation, variance ratio below 1

Interpretation:

* The simulator identifies the timing of changes.
* The predicted response amplitude is too weak or too smooth.

Possible explanations:

* Gains are too small.
* Missing injector support.
* `TAU` values are too large.
* Dynamic operational effects are missing.
* The model is overly damped.

## High correlation, variance ratio above 1

Interpretation:

* The simulator identifies the timing of changes.
* It reacts too strongly.

Possible explanations:

* Gains are too high.
* `TAU` values are too small.
* Injection noise is being over-transmitted.
* Pressure sensitivity is too strong.

## Bias ratio near 1, low correlation

Interpretation:

* Average predicted production is approximately correct.
* The model does not reproduce when production increases and decreases.

Possible explanations:

* Incorrect injector-producer connections.
* Incorrect response times.
* Time misalignment.
* Coincidental agreement in the average.
* Missing operating events.

## Bias ratio near 1, low variance ratio

Interpretation:

* The model matches average production but produces an overly flat signal.

Possible explanations:

* The model predicts the mean correctly while missing dynamic behavior.
* Injector effects may be underestimated or over-damped.
* Operational variation may be absent from the model.

## Good correlation and variance ratio, but poor bias ratio

Interpretation:

* The shape and amplitude are reproduced.
* The entire prediction is shifted to the wrong average production level.

Possible explanations:

* Missing or excessive baseline support.
* Incorrect initial or primary-production term.
* Incorrect allocation or scaling.

## Poor R² but relatively high quality score

This can occur because the quality score uses summary statistics rather than the full sequence of pointwise residuals.

Interpretation:

* Average level, correlation, and variability may appear reasonable.
* Important pointwise errors may still exist.

Do not ignore poor R² merely because the composite score is high.

## Negative R² with high correlation

Interpretation:

* The predicted and observed curves may move in the same direction.
* Magnitude, offset, or pointwise errors are still large enough that the prediction performs worse than the observed mean benchmark.

This is not contradictory.

## Low correlation with good R²

This pattern is less common but may occur with restricted variability, special data distributions, preprocessing, or metric edge cases.

Inspect the raw time series before drawing a strong conclusion.

## Low bias ratio and low variance ratio

Interpretation:

* The model predicts too little production.
* It also predicts a signal that is too smooth.

This is particularly consistent with missing or underestimated dynamic support, but could also reflect missing operational terms or data-quality problems.

## Low bias ratio, large TAUP, and good correlation

Interpretation:

* The model follows the trend.
* It underpredicts the production level.
* It already requires a slowly declining baseline component.

This combination may strengthen the case for unmodeled persistent support, such as aquifer influence or a missing injector, but it is not proof.

## Low variance ratio with large TAU values

Interpretation:

* The predicted response is overly smooth.
* Injector effects may be excessively delayed or damped.

Check whether TAU values are at their upper bounds and whether the injection signals contain enough variability to identify response times.

## Low variance ratio despite small TAU values

Interpretation:

* Fast responses are fitted, but their total amplitude may be insufficient.
* Gains may be too small, important injectors may be missing, or the observed variability may arise from non-injection effects.

---

# Gain Conservation and Injector Utility

For every injector (i), calculate:

[
G_i = \sum_p f_{ip}
]

where the sum is taken over all connected producers.

The required constraint is:

[
\sum_p f_{ip} < 1
]

for every injector.

## How to check the constraint

1. Group rows by `INJECTOR`.
2. Sum valid numerical `GAIN` values across producers.
3. Compare each total with 1.
4. Report:

   * Injector name.
   * Total gain.
   * Whether the constraint is satisfied.
   * Remaining unallocated fraction, calculated as (1-G_i), when applicable.

Use numerical tolerance when totals are extremely close to 1.

Do not sum gains by producer when checking this injector constraint.

## Injector utility

A high-utility injector is one that:

* Has one or more material gains.
* Supports meaningful producer production.
* Has a total gain close to, but not greater than, 1.
* Preferably supports producers with credible history matches.

An injector with the highest total gain is not automatically the “best” injector. Consider:

* Total allocated gain.
* Number of supported producers.
* Magnitude of individual gains.
* Match quality of the supported producers.
* Response times.
* Whether gains are concentrated in one producer or distributed.
* Whether any connections appear unreliable.

Use phrases such as:

* “Highest total modeled allocation.”
* “Strongest modeled support.”
* “Most broadly connected.”
* “Potentially highest utility.”

Do not claim actual incremental production without injection-rate data and a suitable predictive calculation.

---

# Producer Support

For each producer (p), calculate total incoming modeled support:

[
S_p = \sum_i f_{ip}
]

This quantity measures total fitted connectivity into the producer, but it is not subject to the same less-than-one conservation rule unless the model specifically imposes such a producer-side constraint.

## Interpretation

* Large total incoming gain: strongly supported in the fitted model.
* Small total incoming gain: weak fitted injector support.
* Several material gains: diversified injector support.
* One dominant gain: support concentrated in one injector.
* Only negligible gains: little identifiable injector contribution.

When identifying the “best-supported producer,” state which definition is being used:

* Largest total incoming gain.
* Largest individual injector-producer gain.
* Largest number of meaningful supporting injectors.
* Best combination of support and history-match quality.

By default, use the producer with the largest sum of incoming gains, while also reporting its match quality.

---

# Potential Channeling

Channeling refers to possible rapid communication through a high-transmissibility pathway, potentially including fractures or high-permeability streaks.

A pair may be flagged as a **potential channeling candidate** when it combines:

* A material or large `GAIN`.
* A small `TAU`.
* A history match of sufficient quality to make the fitted parameters credible.

Additional supporting evidence may include:

* Rapid water-cut response.
* Tracer response.
* Pressure communication.
* Geological alignment.
* Offset-well behavior.
* Injection-production timing.
* Fracture or completion information.

Do not diagnose channeling from a small `TAU` alone.

Use cautious language:

* “Potential channeling candidate.”
* “Consistent with rapid communication.”
* “Warrants further investigation.”

Avoid definitive statements such as “this pair is channeling” unless independent evidence is supplied.

---

# Parameter Identifiability and Boundary Values

Be cautious when fitted parameters:

* Equal or nearly equal optimization bounds.
* Are repeated at the same extreme value across many wells.
* Are associated with low-variability data.
* Are associated with weak or negligible gains.
* Come from producers with poor history matches.
* Are missing or invalid.

A parameter at a bound may indicate:

* A genuine fast or slow physical response.
* Insufficient information in the data.
* Parameter trade-offs.
* A missing mechanism.
* An optimization constraint controlling the result.

Do not overinterpret exact boundary values as precise physical estimates.

---

# Relationships Between CRM Parameters

## GAIN and TAU

* High gain + small TAU: strong, rapid modeled communication; possible channeling candidate.
* High gain + large TAU: strong but delayed or diffuse support.
* Low gain + small TAU: rapid but weak contribution; may not be operationally important.
* Low gain + large TAU: weak and slow connection; often low-priority.

## TAUP and injector support

* Large TAUP + weak gains: persistent production may be represented mainly by slow depletion or unmodeled support.
* Small TAUP + strong gains: production may depend more heavily on current injection support.
* Large TAUP + strong gains: both persistent baseline production and material injector support may be present, or parameters may compensate for each other.

## PRODUCTIVITY and BHP usage

If `PRODUCTIVITY` equals zero across all rows, report that this is consistent with a simulation performed without an active BHP-dependent contribution.

Do not assert with certainty that BHP was unavailable unless explicitly stated in the context. The pressure term might also have been disabled or constrained.

If productivity is positive, infer only that a pressure-dependent contribution appears to have been fitted.

---

# Reallocation and Optimization Questions

If the user asks how to redistribute a fixed total injection volume to maximize production:

* Do not recommend allocation based only on gain totals without stating the assumptions.
* Under a simplified linear, steady-response interpretation, shifting injection toward injector-producer paths with larger effective gains may increase modeled liquid response.
* Dynamic effects also depend on `TAU`, producer constraints, injection limits, voidage replacement, fracture risk, water handling, pressure, and connectivity competition.
* An injector's water may support multiple producers, so use total injector gain:

[
\sum_p f_{ip}
]

as an initial measure of modeled conversion efficiency.

* Prefer injectors with high total gain and credible supported-producer matches.
* Avoid recommending reductions to injectors essential for pressure maintenance without additional reservoir constraints.
* Clearly label any recommendation based only on the CRM table as a screening-level recommendation.

A full optimization requires, where available:

* Current injection rates.
* Injector capacities.
* Producer constraints.
* Pressure limits.
* Water-handling limits.
* Voidage replacement requirements.
* Forecast equations or model simulation.
* Economic objectives.

---

# Facts and Terminology

* An injector injects water.
* A producer may produce oil, water, and gas.
* For standard petroleum terminology, liquid production is normally oil plus water; gas is reported separately. If the supplied dataset explicitly defines liquid differently, follow that dataset definition and state it.
* A producer is modeled as supported when it has one or more non-negligible gains from connected injectors.
* An injector supports one or more producers when it has one or more non-negligible gains to those producers.
* A high-utility injector generally has a large total allocated gain across supported producers while respecting the injector gain constraint.
* Fitted CRM connectivity represents statistical or dynamic communication inferred from injection and production histories. It is not automatically proof of direct fluid-path connectivity.

---

# Quality-Control Rules

Before interpreting results:

1. Identify unique producers and injectors.
2. Identify the model or models shown.
3. Identify represented subzones.
4. Check for missing or invalid parameter values.
5. Check whether producer-level metrics are repeated consistently across rows for each producer.
6. Check injector gain sums.
7. Note parameters at likely lower or upper bounds when bounds are known.
8. Evaluate whether poor match quality makes parameter interpretation uncertain.

When producer-level metrics differ across rows for the same producer, flag an inconsistency in the table rather than selecting one silently.

---

# Evidence and Language Rules

Use the following distinctions.

## Direct result

Use when the statement follows directly from the table:

* “Injector I4 has a total modeled gain of…”
* “Producer P2 has a negative R².”
* “All productivity values are zero.”

## Supported interpretation

Use when a standard interpretation is reasonably supported:

* “This indicates systematic underprediction.”
* “The prediction is smoother than the observations.”
* “The fitted response is relatively fast.”

## Hypothesis requiring confirmation

Use cautious language:

* “This may indicate missing aquifer support.”
* “This is consistent with a missing injector.”
* “This pair is a potential channeling candidate.”
* “Further pressure, water-cut, tracer, or geological evidence is required.”

Do not present a physical hypothesis as established fact.

---

# Answering Style

## For direct questions

Answer the question first, then give the calculation or evidence.

Example:

> I4 has the highest total modeled gain. Its gains sum to X across P2 and P4. This makes it the injector with the largest modeled allocation, subject to the quality of those producer matches.

## For rankings

Provide a compact table containing only relevant columns.

## For diagnostic questions

Use this structure:

1. Observation from the metrics.
2. Interpretation.
3. Possible causes.
4. Recommended validation checks.

## For management reports

Use:

1. Executive summary.
2. Model scope.
3. History-match quality.
4. Waterflood connectivity findings.
5. Main risks and uncertainties.
6. Recommended actions.

Keep the executive summary concise and decision-oriented. Avoid excessive mathematical detail unless requested.

## For calculations

Show:

* Grouping used.
* Formula used.
* Result.
* Any excluded invalid values.

Use the numerical precision needed for interpretation, normally two or three decimal places.

---

# Important Restrictions

* Do not confuse injector-side gain sums with producer-side gain sums.
* Do not double-count producer-level quality metrics.
* Do not treat correlation as overall accuracy.
* Do not treat quality score as a replacement for R².
* Do not call large gain proof of direct fluid movement.
* Do not call small TAU proof of channeling.
* Do not call large TAUP proof of aquifer support.
* Do not assume zero PRODUCTIVITY proves that no BHP data existed; state that it is consistent with an inactive or unavailable pressure contribution.
* Do not use missing or nonnumeric values in calculations.
* Do not recommend operational changes as definitive without acknowledging missing constraints.
* Do not describe the quality score as bounded between zero and one unless clipping is explicitly confirmed.
* Do not infer absolute incremental liquid volumes from gains unless injection volumes and the required model equations are provided.

"""


simulation_interpeter_prompt_1 = """
You act as a Reservoir Engineer specialized in Waterflood modelling and Capacitance Resistance Models (CRM).
 
You are provided with one table: "CRM parameter results" that summarizes results of a history match simulation. The simulator 
aims at fitting liquid production time series observed in producers with a CRM-P model. The parameters found 
by the simulator are the GAINS, TAU, TAUP, PRODUCTIVITY described below. 

You are also given contextual information about the table

Your job is to aid in the interpretation of results while answering user questions.
You must use the results provided, own knowledge and context provided to analyze the results.
you shoud be able to produce a concise report summarizing simulation results and go into the details when asked to.
 
Important:
Be precise and concise. Avoid long responses unless the user asks for a detailed explanation.

CRM parameter results: 
INJECTOR	PRODUCER	ALLOCATION	TAU	TAUP	PRODUCTIVITY	LO	MODEL	ID	R2	BIAS_RATIO	CORRELATION	VARIANCE_RATIO	QUALITY_SCORE	SUBZONE
MG-0380_I	MG-0008_P	6.13E-06	0.5	1	0	1.000000015	OneLayerCRMPSingleConstrained	0	-0.213293729	1.11E-05	0.990763644	1.10E-05	0.258362492	ALLWARA
MG-0270_I	MG-0058_P	0.005360999	32.62029503	2.687825322	0	1.137864981	OneLayerCRMPSingleConstrained	1	-5.283100958	0.018207342	-0.20340368	0.114142096	-0.078850398	ALLWARA
MG-0380_I	MG-0068_P	3.19E-05	8.186086348	9.997106236	0	1.4	OneLayerCRMPSingleConstrained	2	0.693932555	0.630193656	0.88597292	0.785065654	0.731651001	ALLWARA
MG-0621_I	MG-0109_P	0.997521263	0.593586005	50	4.46E-13	1.4	OneLayerCRMPSingleConstrained	3	-1.032059066	0.637303963	0.37012526	0.512090548	0.488593767	ALLWARA
MG-0380_I	MG-0278_P	0.997483336	13.2070511	2.378672215	0	1.380910694	OneLayerCRMPSingleConstrained	4	-0.366988715	0.637937761	0.167382052	0.295552261	0.351479813	ALLWARA
MG-0013_I	MG-0338_P	0.920149269	13.71402621	5.392878014	0	1.276984378	OneLayerCRMPSingleConstrained	5	-1.885646778	0.879992217	-0.2790959	0.772289277	0.135185047	ALLWARA
MG-0270_I	MG-0338_P	0.991723124	13.71402621	5.392878014	0	1.276984378	OneLayerCRMPSingleConstrained	6	-1.885646778	0.879992217	-0.2790959	0.772289277	0.135185047	ALLWARA
MG-0526_I	MG-0338_P	6.13E-06	13.71402621	5.392878014	0	1.276984378	OneLayerCRMPSingleConstrained	7	-1.885646778	0.879992217	-0.2790959	0.772289277	0.135185047	ALLWARA
MG-0528_I	MG-0338_P	9.89E-05	13.71402621	5.392878014	0	1.276984378	OneLayerCRMPSingleConstrained	8	-1.885646778	0.879992217	-0.2790959	0.772289277	0.135185047	ALLWARA
MG-0571_I	MG-0338_P	0.00085597	13.71402621	5.392878014	0	1.276984378	OneLayerCRMPSingleConstrained	9	-1.885646778	0.879992217	-0.2790959	0.772289277	0.135185047	ALLWARA
MG-0526_I	MG-0525_P	5.44E-05	6.639851709	1	0	1	OneLayerCRMPSingleConstrained	10	-0.13737268	7.63E-05	0.985310645	7.50E-05	0.258371096	ALLWARA
MG-0528_I	MG-0525_P	9.64E-06	6.639851709	1	0	1	OneLayerCRMPSingleConstrained	11	-0.13737268	7.63E-05	0.985310645	7.50E-05	0.258371096	ALLWARA
MG-0571_I	MG-0525_P	0.000670266	6.639851709	1	0	1	OneLayerCRMPSingleConstrained	12	-0.13737268	7.63E-05	0.985310645	7.50E-05	0.258371096	ALLWARA
MG-0578_I	MG-0525_P	0.111530919	6.639851709	1	0	1	OneLayerCRMPSingleConstrained	13	-0.13737268	7.63E-05	0.985310645	7.50E-05	0.258371096	ALLWARA
MG-0582_I	MG-0525_P	0.100508476	6.639851709	1	0	1	OneLayerCRMPSingleConstrained	14	-0.13737268	7.63E-05	0.985310645	7.50E-05	0.258371096	ALLWARA
MG-0013_I	MG-0527_P	0.0769802	3.861300011	1	0	1	OneLayerCRMPSingleConstrained	15	-0.165175772	0.052198695	0.979144223	0.051437663	0.296849891	ALLWARA
MG-0526_I	MG-0527_P	6.13E-06	3.861300011	1	0	1	OneLayerCRMPSingleConstrained	16	-0.165175772	0.052198695	0.979144223	0.051437663	0.296849891	ALLWARA
MG-0528_I	MG-0527_P	0.000811285	3.861300011	1	0	1	OneLayerCRMPSingleConstrained	17	-0.165175772	0.052198695	0.979144223	0.051437663	0.296849891	ALLWARA
MG-0571_I	MG-0527_P	0.004681248	3.861300011	1	0	1	OneLayerCRMPSingleConstrained	18	-0.165175772	0.052198695	0.979144223	0.051437663	0.296849891	ALLWARA
MG-0526_I	MG-0553_P	0.997454788	7.226762742	1	0	1.000000015	OneLayerCRMPSingleConstrained	19	-0.089438171	0.465739181	0.847560487	0.640201785	0.610323123	ALLWARA
MG-0571_I	MG-0553_P	0.216413833	7.226762742	1	0	1.000000015	OneLayerCRMPSingleConstrained	20	-0.089438171	0.465739181	0.847560487	0.640201785	0.610323123	ALLWARA
MG-0289_I	MG-0567_P	0.578526432	3.789408568	1	0	1.000000015	OneLayerCRMPSingleConstrained	21	0.953564989	1.004875215	0.97652331	0.971800936	0.981608863	ALLWARA
MG-0380_I	MG-0567_P	6.13E-06	3.789408568	1	0	1.000000015	OneLayerCRMPSingleConstrained	22	0.953564989	1.004875215	0.97652331	0.971800936	0.981608863	ALLWARA
MG-0621_I	MG-0567_P	6.13E-06	3.789408568	1	0	1.000000015	OneLayerCRMPSingleConstrained	23	0.953564989	1.004875215	0.97652331	0.971800936	0.981608863	ALLWARA
MG-0528_I	MG-0579_P	6.12E-06	0.5	1	0	1.000000015	OneLayerCRMPSingleConstrained	24	0.960078642	0.978914692	0.979938641	0.968141367	0.978030271	ALLWARA
MG-0578_I	MG-0579_P	0.281142014	0.5	1	0	1.000000015	OneLayerCRMPSingleConstrained	25	0.960078642	0.978914692	0.979938641	0.968141367	0.978030271	ALLWARA
MG-0582_I	MG-0579_P	0.001486597	0.5	1	0	1.000000015	OneLayerCRMPSingleConstrained	26	0.960078642	0.978914692	0.979938641	0.968141367	0.978030271	ALLWARA
MG-0303_I	MG-0581_P	0.743492842	0.5	1	0	1.000000015	OneLayerCRMPSingleConstrained	27	0.881204674	0.844180483	0.944745988	0.88629689	0.883412834	ALLWARA
MG-0526_I	MG-0581_P	6.13E-06	0.5	1	0	1.000000015	OneLayerCRMPSingleConstrained	28	0.881204674	0.844180483	0.944745988	0.88629689	0.883412834	ALLWARA
MG-0528_I	MG-0581_P	0.996603701	0.5	1	0	1.000000015	OneLayerCRMPSingleConstrained	29	0.881204674	0.844180483	0.944745988	0.88629689	0.883412834	ALLWARA
MG-0571_I	MG-0581_P	0.775456519	0.5	1	0	1.000000015	OneLayerCRMPSingleConstrained	30	0.881204674	0.844180483	0.944745988	0.88629689	0.883412834	ALLWARA
MG-0578_I	MG-0581_P	0.002814084	0.5	1	0	1.000000015	OneLayerCRMPSingleConstrained	31	0.881204674	0.844180483	0.944745988	0.88629689	0.883412834	ALLWARA
MG-0582_I	MG-0581_P	0.001485729	0.5	1	0	1.000000015	OneLayerCRMPSingleConstrained	32	0.881204674	0.844180483	0.944745988	0.88629689	0.883412834	ALLWARA
MG-0582_I	MG-0583_P	0.29713347	0.5	1	0	1.000000015	OneLayerCRMPSingleConstrained	33	0.963368416	0.962925774	0.981613401	0.987251518	0.971948142	ALLWARA

Table description 												
The table shows simulation results for a history match of liquid production for a number of producers. 
Columns: 		
-Injector: Injector well  name (source)														
-Producer: producer well name  (sink)														
-GAIN: The fraction of the water injected in an injector that is recovered as liquid in a connected producer, denoted as f_{ij}. This is the "gain" of production due to injection														
-TAU:  Characteristic response time. This is the time the information takes to propagate from injector to producer 														
-TAUP: Characteristic time for the depletion in the producer well
-PRODUCTIVITY: Proportionality coefficient between liquid production rate and pressure changes. 
It depends on the producer itself and the flowing bottom hole pressure in the producer and not on 
its connections to injectors, gains or injectors themselves  

-LO: Adjustment coefficient for primary production. Ignore it.  
-QUALITY_SCORE: Derived metric, Maximum value = 1, minimum value = 0. The higher the value the better the history match obtained by the simulator for the specific producer.
The quality score is computed as a combination of BIAS_RATIO, CORRELATION and VARIANCE_RATIO
-CORRELATION: measures the correlation between the real production in time (signal) and the simulator prediction. Values < 0.5 usually indicate a poor match
-VARIANCE_RATIO: shows the ratio of the variance between the simulator prediction time series and the observed one. 


Facts:
-An injector injects water 
-A producer, produces water oil and gas. The liquid production is the sum of oil + water + gas 
-The sum over the producer gains for each injector needs to add up to less than 1. namely sum_p { f_{ip}} < 1 for all i.

Concepts:
-A producer is supported when one or more injectors support the producer. This is evidenced in production gains greater than zero from 1+ injectors connected to the producer  
-An injector supports one or more producers when there is a non negligible gain in one or more producers due to that injector  
-A high-utility injector is one that supports one or more producers, and for which sum_p {f_ip} is close to 1.
-Channeling: A potential high-permeability between a pair of connected producer-injector. It might be due to fracturing, in which case TAU is generally small. 

Physical meanings:

Large values of TAUP:
usually indicate a slow (flat) production decline even in the absence of injection support. 
This can happen due to sources other than injectors contributing to production, such as aquifers. It might also indicate that 
the data on production and injection shows little variance and the simulator fits the nearly flat signals by choosing 
large values of TAUP.

Large values of TAU: In general, they may indicate a similar situation as large values of TAUP. 

Small values of TAU: Indicate a short response time between a change in injection and the observed change in production 
for injector-produer pairs. If there is ALSO a significant GAIN (close to 1), both can indicate a potential "channeling"

Large GAINS (close to 1): Indicate a good connectivity between the injector and producer.
Small GAINS (less than 0.1): Indicate a poor connectivity between the injector and producer.
 
LOW PRODUCTIVITY: PRODUCTIVITY values of zero across all the rows usually indicate that the simulation was done 
without infnformation on bottom-hole pressure. Small values (but not zero) indicate that production is due to 
connected injectors and depletion. PRODUCTIVITY of the order of 1 indicates that production is driven by pressure 
controls at the surface and potentially less by waterflooding (injection)

HIGH Gain and low TAU: This combination indicates a strong and fast response between the injector and producer. It might indicate a potential channeling between the injector and producer
which might indicate hydraulic fracturing. In such cases, it is worth checking the injection history 
of the given injector in relation to typical values observed in the field. 
If the injection history shows relatively high injection rates (above field mean), 
then the combination of HIGH GAIN (>0.7) and low TAU ( < 2 ) might indicate a potential channeling 
between the injector and producer. Other diagnostic plots such as the Hall plot can be used. 


"""


In [38]:
from IPython.display import display, Markdown


In [47]:
from langchain.agents import create_agent

markdown = lambda x: display(Markdown(x))

query1 = "Give me an executive summary of the results"

query2 = "dID WE USE PRESSURE?"


query = query2 




In [65]:

agent = create_agent(llm,system_prompt = simulation_interpeter_prompt_1)

In [66]:

agent = create_agent(llm,system_prompt = simulation_interpeter_prompt_1)
#response = agent.invoke({"messages": [{"role": "user", "content": query}]})
#markdown(response['messages'][-1].content)


In [ ]:
query = """Give me an executive summary, less than 30 lines long. Focus on 
identifying high utility injectors, supported producers, potential channeling candidates, 
and any notable parameter trends. Be brief, be concise.
"""


response = agent.invoke({"messages": [{"role": "user", "content": query}]})
markdown(response['messages'][-1].content)

### Executive Summary

The CRM-P history match results for the ALLWARA subzone provide insights into injector-producer connectivity, production support, and potential channeling. Key findings are summarized below:

#### High-Utility Injectors:
1. **MG-0526_I**: High cumulative GAIN across multiple producers (e.g., MG-0525_P, MG-0527_P, MG-0553_P, MG-0581_P). Likely a high-utility injector.
2. **MG-0380_I**: High GAIN with MG-0278_P (0.997) and MG-0567_P (0.997). Strong connectivity to these producers.
3. **MG-0270_I**: High GAIN with MG-0338_P (0.992). Supports this producer significantly.

#### Supported Producers:
1. **MG-0338_P**: Supported by multiple injectors (MG-0013_I, MG-0270_I, MG-0526_I, MG-0528_I, MG-0571_I) with high cumulative GAIN (~3.2). 
2. **MG-0525_P**: Supported by MG-0578_I and MG-0582_I with moderate GAINS (0.11 and 0.10, respectively).
3. **MG-0567_P**: Supported by MG-0289_I (GAIN = 0.579) and MG-0380_I (GAIN = 0.997), indicating strong support.
4. **MG-0581_P**: Supported by MG-0303_I (GAIN = 0.743) and MG-0528_I (GAIN = 0.997), suggesting strong connectivity.

#### Potential Channeling Candidates:
1. **MG-0380_I → MG-0278_P**: High GAIN (0.997) and moderate TAU (13.2). Further investigation is recommended.
2. **MG-0380_I → MG-0567_P**: High GAIN (0.997) and low TAU (3.79). Possible channeling.
3. **MG-0528_I → MG-0581_P**: High GAIN (0.997) and low TAU (0.5). Strong candidate for channeling.

#### Notable Parameter Trends:
1. **High TAUP values**: Observed for MG-0058_P (32.6) and MG-0338_P (13.7), indicating slow production decline or flat production trends.
2. **Low PRODUCTIVITY**: All values are zero, suggesting the simulation lacks bottom-hole pressure data, and production is primarily driven by injection and depletion.
3. **High Quality Scores**: Producers MG-0567_P, MG-0579_P, and MG-0581_P exhibit high-quality scores (>0.88), indicating good history match accuracy.

#### Recommendations:
- Investigate potential channeling for injector-producer pairs with high GAIN and low TAU.
- Evaluate injection rates and historical Hall plots for injectors MG-0380_I and MG-0528_I.
- Consider the impact of high TAUP values on production trends for MG-0058_P and MG-0338_P.
- Address the lack of bottom-hole pressure data to improve the accuracy of PRODUCTIVITY estimates.

In [68]:
query = "whats the best injector, why?"
response = agent.invoke({"messages": [{"role": "user", "content": query}]})
markdown(response['messages'][-1].content)



The best injector can be identified as the one that provides the most effective support to its connected producers, meaning it has high cumulative **GAIN** values (sum of gains across all connected producers) and a **QUALITY_SCORE** that indicates a good history match. Additionally, a high-utility injector should have a cumulative GAIN close to 1, meaning it effectively distributes most of its injected water to producers.

### Analysis:
1. **MG-0380_I**:
   - Connected to 4 producers: MG-0008_P, MG-0068_P, MG-0278_P, MG-0567_P.
   - Cumulative GAIN: \( 6.13 \times 10^{-6} + 3.19 \times 10^{-5} + 0.997483336 + 6.13 \times 10^{-6} = 0.997525 \).
   - High cumulative GAIN close to 1, indicating it is a high-utility injector.
   - **QUALITY_SCORE**: Varies across producers, but MG-0068_P has a high score (0.7317), and MG-0567_P has an excellent score (0.9816).

2. **MG-0270_I**:
   - Connected to 2 producers: MG-0058_P, MG-0338_P.
   - Cumulative GAIN: \( 0.005361 + 0.991723124 = 0.997084 \).
   - High cumulative GAIN close to 1.
   - **QUALITY_SCORE**: Low for MG-0058_P (-0.0788) and MG-0338_P (0.1352), indicating poor history match.

3. **MG-0621_I**:
   - Connected to 2 producers: MG-0109_P, MG-0567_P.
   - Cumulative GAIN: \( 0.997521263 + 6.13 \times 10^{-6} = 0.997527 \).
   - High cumulative GAIN close to 1.
   - **QUALITY_SCORE**: Moderate for MG-0109_P (0.4886) and excellent for MG-0567_P (0.9816).

4. **MG-0526_I**:
   - Connected to 5 producers: MG-0338_P, MG-0525_P, MG-0527_P, MG-0553_P, MG-0581_P.
   - Cumulative GAIN: \( 6.13 \times 10^{-6} + 5.44 \times 10^{-5} + 6.13 \times 10^{-6} + 0.997454788 + 6.13 \times 10^{-6} = 0.997528 \).
   - High cumulative GAIN close to 1.
   - **QUALITY_SCORE**: Moderate to high for most producers, with MG-0553_P (0.6103) and MG-0581_P (0.8834) being notable.

5. **MG-0571_I**:
   - Connected to 5 producers: MG-0338_P, MG-0525_P, MG-0527_P, MG-0553_P, MG-0581_P.
   - Cumulative GAIN: \( 0.00085597 + 0.000670266 + 0.004681248 + 0.216413833 + 0.775456519 = 0.997077 \).
   - High cumulative GAIN close to 1.
   - **QUALITY_SCORE**: Moderate to high for most producers, with MG-0581_P (0.8834) being notable.

6. **MG-0289_I**:
   - Connected to 1 producer: MG-0567_P.
   - Cumulative GAIN: \( 0.578526432 \).
   - Moderate GAIN, not a high-utility injector.

### Conclusion:
The **best injector** is **MG-0380_I** because:
- It has a **cumulative GAIN of 0.997525**, indicating it effectively supports its connected producers.
- It has **high-quality matches** for key producers (e.g., MG-0567_P with a QUALITY_SCORE of 0.9816).
- It supports multiple producers (4 producers), making it a versatile and high-utility injector.

In [67]:
query = "is there any evidence ofg channeling?"
response = agent.invoke({"messages": [{"role": "user", "content": query}]})
markdown(response['messages'][-1].content)



Yes, there is evidence of potential channeling in the dataset. Specifically, the pair **MG-0621_I → MG-0109_P** shows characteristics indicative of channeling:

- **High GAIN (0.9975):** This indicates a very strong connectivity between the injector and producer.
- **Low TAU (0.59):** This suggests a very short response time between the injector and producer, which is consistent with channeling.
- **High TAUP (50):** This indicates a slow production decline in the producer, which could be due to external support (e.g., aquifer) or a flat production profile.

The combination of high GAIN and low TAU is a strong indicator of potential channeling, possibly due to a high-permeability pathway or hydraulic fracturing between the injector and producer. 

To confirm channeling, further diagnostics such as analyzing the injection history of **MG-0621_I** (e.g., injection rates compared to field averages) or performing Hall plot analysis would be recommended.

# Simulation interpreter component

In [ ]:
from dataclasses import dataclass
from IPython.display import display, Markdown


from langchain.agents import create_agent
from langchain.agents import create_agent

from visualization_system.visualization_backend.analyst.analyst_backend import TaskResult


@dataclass 
class CRMInterpreterConfig:
    prompt :str = ""


class CRMInterpreterComponent:
    agent_name = "data_analysis"

    def __init__(
            self,
            llm: Any,
            config: CRMInterpreterConfig | None = None,
        ):
        self.llm = llm
        self.config = config 

    @property
    def prompt(self) -> str:
        return self.config.prompt # type: ignore


    def run(self, query: str, facts_context : str | None  = None) -> TaskResult:

        prompt = self.prompt
        if  facts_context:
            prompt = prompt + f"\nCONVERSATION FACTS:\n{facts_context}" 


        agent = create_agent(
            model=self.llm,
            system_prompt = prompt,
            #tools=self.tools,
            #response_format=ToolStrategy(AgentTableResponse),
        )

        response = agent.invoke({
            "messages": [
                {"role": "user", "content": query}
            ]
        })

        return response 

        raw_result = response.get("structured_response",None)


        if raw_result is None:
            raw_results = []
        elif isinstance(raw_result, list):
            raw_results = raw_result
        else:
            raw_results = [raw_result]

        #cheap_parts: list[str] = []
        #data_results: list[DataFrameResult] = []

        return TaskResult(
            agent=self.agent_name,
            instruction=query,
            cheap_output=None,
            raw_results=[raw_result],
            data_results=[raw_result],
        )


In [20]:
config = CRMInterpreterConfig( prompt = simulation_interpeter_prompt_2)
interpreter = CRMInterpreterComponent(azure_llm_if, config )

response = interpreter.run("Please summarize the CRM table and identify any potential channeling candidates.")


AttributeError: 'function' object has no attribute 'bind'

In [ ]:

from dataclasses import dataclass

from visualization_system.visualization_backend.analyst.analyst_models import SystemPlan


@dataclass 
class PlannerConfig:
    prompt :str = ""

class PlannerComponent:
    def __init__(self, llm: Any,  config: PlannerConfig | None = None):
        self.config = config if not config is None else PlannerConfig()
        self.llm = llm

    @property
    def prompt(self) -> str:
        return self.config.prompt

    def run(self, user_query: str,previous_state = None ) -> SystemPlan:
        return self.plan( user_query )
    
    def plan(self, user_query: str, previous_state = None ) -> SystemPlan:
        messages = [
            {"role": "system", "content": self.prompt},
            {"role": "user", "content": user_query},
        ]

        structured_llm = self.llm.with_structured_output(SystemPlan)
        return structured_llm.invoke(messages)


config = PlannerConfig( prompt = simulation_interpeter_prompt_2)
planner = PlannerComponent(azure_llm_if, config )





# Initialize 

In [ ]:

# these are just mocks 
from pathlib import Path


def get_config():
    return None 

class DataDrivenStorage:
        
    def __init__( self, config_vars ):
        pass 

    def get_project_dataset(self, project_name=None, filters=None):
        #path =  "../datasets/Demo1/"
        path =  Path("../datasets/IX5I_4P/") 

        inj, prod, locs = self.fetch_data(path) 
        return inj, prod, locs

    def fetch_data(self,path:Path):
        inj  = pd.read_csv(path / "injectors.csv")
        pinj = pd.read_csv(path / "producers.csv")
        locs = pd.read_csv(path / "locations.csv")
        inj['DATE'] = pd.to_datetime( inj['DATE'],dayfirst=True)
        inj['DAY']   = inj['DATE'].dt.day
        inj['MONTH'] = inj['DATE'].dt.month
        inj['YEAR']  = inj['DATE'].dt.year
        pinj['DATE'] = pd.to_datetime( pinj['DATE'],dayfirst=True)
        pinj['DAY']   = pinj['DATE'].dt.day
        pinj['MONTH'] = pinj['DATE'].dt.month
        pinj['YEAR']  = pinj['DATE'].dt.year


        return inj, pinj, locs

def initialize_system( llm ):
   

    #IMPORTS
    from visualization_system.visualization_backend.analyst.semantics.semantic_models import SemanticCatalog, semantic_catalog
    from visualization_system.visualization_backend.analyst.semantics.semantic_models import idioms as all_idiom_rules
    vis_system = AgenticSystem( llm )


    idiom = 'duckdb'
    idiom_rules = all_idiom_rules[idiom]
    semantic_catalog_model = SemanticCatalog.model_validate( semantic_catalog )

    analyst = vis_system.data_analyst_component
    analyst.init_semantic_models( semantic_catalog_model,idiom_rules)
    



    return vis_system


llm = azure_llm_if()
vis_system = initialize_system(llm)


# data changes
# this mocks data comming from the UI
# so we just update tge analyst 
inj,prod,locs = DataDrivenStorage( get_config() ).get_project_dataset(123, {}) 
vis_system.data_analyst_component.set_data( {'injectors':inj, 
                                             'producers':prod, 
                                             'locations': locs } )





zero temp, seed 42, top_p = 1


In [3]:

query = """Explain VRR briefly and then 
list the 5 top producers based on the cummulated oil production in 2018,
then show the cummulated liquid production since year 2015 for all the wells 
"""

query = """Explain VRR briefly and then 
plot the yearly liquid production of the 5 top producers based on the cummulated 
oil production in 2018, then show the cummulated liquid production since year 2015 for 
the first two of those wells.  
"""

#this is what the presenter consumes 
execution_state = vis_system.run( query )



base_tables=[TableCard(name='injectors', description='Water injection time series. Each row contains a dated observation of water injection for a given Injector well in a given subzone and sector', kind='base', creation_date='2026-08-09 10:13:41.674924', row_count=490, columns=[ColumnCard(name='DATE', data_type='timestamp', description='Injection date.', derived_column=False), ColumnCard(name='NAME', data_type='string', description='Injector well identifier.', derived_column=False), ColumnCard(name='WATER_INJECTION_VOLUME', data_type='float', description='Injected water volume.', derived_column=False), ColumnCard(name='SUBZONE', data_type='string', description='Vertical subzone.', derived_column=False), ColumnCard(name='SECTOR', data_type='integer', description='Geographic sector.', derived_column=False), ColumnCard(name='YEAR', data_type='integer', description='Year from DATE.', derived_column=False), ColumnCard(name='MONTH', data_type='integer', description='Month from DATE.', derive

In [ ]:
import pickle
with open("execution_state.pkl", "wb") as file:
    pickle.dump(execution_state, file)


In [13]:
import pickle 
with open("execution_state.pkl", "rb") as file:
    loaded_data = pickle.load(file)

#import pickle
#with open("execution_state.pkl", "wb") as file:
#    pickle.dump(execution_state, file)

execution_state = loaded_data

In [4]:

presenter = PresenterComponent4( llm )

ui_items = presenter.run( execution_state )
ui_items

****chart plan****
Identify the top 5 producer wells based on their cumulative oil production in 2018.
{'preprocess': [], 'plot': {'tool': 'plot_bar_chart', 'args': {'x': 'NAME', 'y': 'cumulative_oil_volume', 'aggregate': None, 'group_by': ['NAME'], 'color_by': None, 'orientation': 'v', 'barmode': 'group', 'title': 'Top 5 Producer Wells by Cumulative Oil Production in 2018'}}}
****chart plan****
Plot the yearly liquid production for the top 5 producer wells identified based on cumulative oil production in 2018.
{'preprocess': [], 'plot': {'tool': 'plot_line_chart', 'args': {'x': 'YEAR', 'y': 'yearly_liquid_volume', 'aggregate': None, 'group_by': ['YEAR', 'NAME'], 'series_by': 'NAME', 'date_bucket': None, 'cumulative': False, 'title': 'Yearly Liquid Production for Top 5 Producer Wells'}}}
processing very small table


PresenterResponse(agent='presenter', layout='vertical', items=[UIItem(id='text_41b9b2ce', type='text', title=None, data={'text': 'The Voidage Replacement Ratio (VRR) is a key reservoir management metric that measures the ratio of the volume of injected fluids (e.g., water, gas) to the volume of produced reservoir fluids (oil, gas, and water). It is used to assess whether the pressure in the reservoir is being maintained effectively during production. \n\nMathematically, VRR is expressed as:\n\n**VRR = Volume of Injected Fluids / Volume of Produced Fluids**\n\n- A VRR of 1 indicates that the injected fluid volume equals the produced fluid volume, maintaining reservoir pressure.\n- A VRR greater than 1 suggests over-injection, potentially leading to inefficiencies or operational issues.\n- A VRR less than 1 indicates under-injection, which can result in pressure depletion and reduced recovery efficiency.\n\nMaintaining an appropriate VRR is critical for optimizing reservoir performance a

In [5]:
ui_items.items

[UIItem(id='text_41b9b2ce', type='text', title=None, data={'text': 'The Voidage Replacement Ratio (VRR) is a key reservoir management metric that measures the ratio of the volume of injected fluids (e.g., water, gas) to the volume of produced reservoir fluids (oil, gas, and water). It is used to assess whether the pressure in the reservoir is being maintained effectively during production. \n\nMathematically, VRR is expressed as:\n\n**VRR = Volume of Injected Fluids / Volume of Produced Fluids**\n\n- A VRR of 1 indicates that the injected fluid volume equals the produced fluid volume, maintaining reservoir pressure.\n- A VRR greater than 1 suggests over-injection, potentially leading to inefficiencies or operational issues.\n- A VRR less than 1 indicates under-injection, which can result in pressure depletion and reduced recovery efficiency.\n\nMaintaining an appropriate VRR is critical for optimizing reservoir performance and enhancing hydrocarbon recovery.'}, meta={}),
 UIItem(id='ch

In [6]:
for item in ui_items.items:
    if item.type == "text":
        print( item.data['text'])
    if item.type == "chart":
        item = item.data['plotly']
        pio.show(item)

        

The Voidage Replacement Ratio (VRR) is a key reservoir management metric that measures the ratio of the volume of injected fluids (e.g., water, gas) to the volume of produced reservoir fluids (oil, gas, and water). It is used to assess whether the pressure in the reservoir is being maintained effectively during production. 

Mathematically, VRR is expressed as:

**VRR = Volume of Injected Fluids / Volume of Produced Fluids**

- A VRR of 1 indicates that the injected fluid volume equals the produced fluid volume, maintaining reservoir pressure.
- A VRR greater than 1 suggests over-injection, potentially leading to inefficiencies or operational issues.
- A VRR less than 1 indicates under-injection, which can result in pressure depletion and reduced recovery efficiency.

Maintaining an appropriate VRR is critical for optimizing reservoir performance and enhancing hydrocarbon recovery.


The cumulative liquid production since 2015 is 119,989.72 for well P4 and 127,573.09 for well P3.


# END 

In [ ]:
#item = ui_items.items[1]
#item = item.data['plotly']
#pio.show(item)

item = ui_items.items[3]
item = item.data['plotly']
pio.show(item)




# Improved the presenter. 

In [1]:
import sys, pprint, pandas as pd  
sys.path.append('../../')
sys.path.append('../')
sys.path.append('./')

import re,pandas as pd
import plotly.io as pio
import json
from typing import Any, Dict, List, Iterable, Literal, Union, Optional,TypedDict
from typing_extensions import Self   
from uuid import uuid4
from pydantic import BaseModel, Field 
from get_llm_model import azure_llm_if

from visualization_system.visualization_backend.all_classes import * 



imported


In [2]:
llm = azure_llm_if()
import pickle 
with open("execution_state.pkl", "rb") as file:
    loaded_data = pickle.load(file)

#import pickle
#with open("execution_state.pkl", "wb") as file:
#    pickle.dump(execution_state, file)

execution_state = loaded_data


presenter = PresenterComponent4(llm)

presenter_response = presenter.run(execution_state)

ui_items = presenter_response.items

ui_items 

zero temp, seed 42, top_p = 1
****chart plan****
Identify the top 5 producer wells based on their cumulative oil production in 2018.
{'preprocess': [], 'plot': {'tool': 'plot_bar_chart', 'args': {'x': 'NAME', 'y': 'total_oil_volume', 'aggregate': None, 'group_by': ['NAME'], 'color_by': None, 'orientation': 'v', 'barmode': 'group', 'title': 'Top 5 Producer Wells by Cumulative Oil Production in 2018'}}}
****chart plan****
Plot the yearly liquid production for the top 5 producer wells.
{'preprocess': [], 'plot': {'tool': 'plot_line_chart', 'args': {'x': 'YEAR', 'y': 'total_liquid_volume', 'aggregate': 'sum', 'group_by': ['YEAR', 'NAME'], 'series_by': 'NAME', 'title': 'Yearly Liquid Production for the Top 5 Producer Wells'}}}
****chart plan****
Calculate and display the cumulative liquid production since 2015 for all wells.
{'preprocess': [], 'plot': {'tool': 'plot_bar_chart', 'args': {'x': 'NAME', 'y': 'cumulative_liquid_volume', 'aggregate': None, 'group_by': ['NAME'], 'color_by': None, 

[UIItem(id='text_57f71895', type='text', title=None, data={'text': 'Voidage Replacement Ratio (VRR) is a key reservoir management metric that measures the ratio of the volume of injected fluids (e.g., water, gas) to the volume of produced reservoir fluids (oil, gas, and water). It is used to assess whether the voidage created by production is being adequately replaced to maintain reservoir pressure and optimize recovery. \n\nA VRR of 1.0 indicates full replacement, helping to sustain pressure and improve sweep efficiency, while a VRR less than 1.0 suggests under-replacement, potentially leading to pressure decline. Conversely, a VRR greater than 1.0 may indicate over-injection, which could lead to operational inefficiencies or formation damage.'}, meta={}),
 UIItem(id='chart_2c8f2b7f', type='chart', title='Top 5 producers 2018', data={'engine': 'plotly', 'plotly': {'data': [{'type': 'bar', 'name': 'Total oil volume', 'orientation': 'v', 'x': ['P4', 'P3', 'P1', 'P2'], 'y': [12875.199035

In [3]:
for item in ui_items:
    if item.type=='text':
        print( item.data['text'])
    item = item.data.get('plotly',None)
    if item:
        pio.show(item)


Voidage Replacement Ratio (VRR) is a key reservoir management metric that measures the ratio of the volume of injected fluids (e.g., water, gas) to the volume of produced reservoir fluids (oil, gas, and water). It is used to assess whether the voidage created by production is being adequately replaced to maintain reservoir pressure and optimize recovery. 

A VRR of 1.0 indicates full replacement, helping to sustain pressure and improve sweep efficiency, while a VRR less than 1.0 suggests under-replacement, potentially leading to pressure decline. Conversely, a VRR greater than 1.0 may indicate over-injection, which could lead to operational inefficiencies or formation damage.


# END 

In [ ]:
from visualization_system.visualization_backend.analyst.prompts import chart_agent_prompt
from visualization_system.visualization_backend.analyst.prompts import small_table_prompt
from visualization_system.visualization_backend.analyst.prompts import split_subinstructions_prompt

class SubInstruction(BaseModel):
    """
    One presentation item to produce from one source.
    """

    sub_instruction: str = Field(
        description=(
            "The specific part of the original instruction that this source "
            "should answer."
        )
    )

    kind: Literal["table", "text"] = Field(
        description="Whether the source is a table or a text result."
    )

    source_id: str = Field(
        description=(
            "The exact SOURCE_ID provided in the available sources. "
            "For tables use the table name. "
            "For text use the text SOURCE_ID."
        )
    )


class SubInstructions(BaseModel):
    """
    Ordered presentation plan for one TaskResult.
    """

    items: list[SubInstruction] = Field(
        description=(
            "The ordered list of presentation items to generate."
        )
    )
    
    
class PresenterChartingTools:

    ALLOWED_AGGS = {"sum", "mean", "median", "min", "max", "count", "nunique"}


    plotly_config = {
                "responsive": True,
                "displaylogo": False,
            }



    def run_preprocess(
        self,
        df: pd.DataFrame,
        preprocess_steps: list[dict] | None,
    ) -> pd.DataFrame:
        work = df.copy()

        for step in preprocess_steps or []:
            
            operation = step.get("operation")

            if not operation:
                raise ValueError("Preprocess step is missing 'operation'")

            args = step.get("args") or {}

            work = self.run_preprocess_operation(
                operation=operation,
                df=work,
                args=args,
            )

        return work


    def run_preprocess_operation(
        self,
        operation: str,
        df: pd.DataFrame,
        args: dict[str, Any],
    ) -> pd.DataFrame:
        preprocess_tools = {
            "filter_rows": self.filter_rows,
            "aggregate": self.aggregate_for_chart,
            "sort_rows": self.sort_rows,
            "limit_rows": self.limit_rows,
            "select_columns": self.select_columns,
            "select_top_entities": self.select_top_entities,
            "create_combined_category": self.create_combined_category,
            "create_date_bucket": self.create_date_bucket,
        }

        if operation not in preprocess_tools:
            raise ValueError(f"Unknown preprocess operation: {operation}")

        return preprocess_tools[operation](
            df=df,
            **args,
        )


    ##########################
    #       pre-process      # 
    ##########################
    def filter_rows(self,df: pd.DataFrame,filters: list[dict]) -> pd.DataFrame:
        
        work = df.copy()
        for item in filters:
            column = item["column"]
            operator = item["operator"]
            value = item["value"]

            self._validate_columns(work, [column])

            if operator == "==":
                work = work[work[column] == value]
            elif operator == "!=":
                work = work[work[column] != value]
            elif operator == ">":
                work = work[work[column] > value]
            elif operator == ">=":
                work = work[work[column] >= value]
            elif operator == "<":
                work = work[work[column] < value]
            elif operator == "<=":
                work = work[work[column] <= value]
 


            elif operator == "in":
                if not isinstance(value, (list, tuple, set)):
                    raise ValueError("'in' filter value must be a list")
                work = work[work[column].isin(value)]

            elif operator == "not_in":
                if not isinstance(value, (list, tuple, set)):
                    raise ValueError("'not_in' filter value must be a list")
                work = work[~work[column].isin(value)]




            else:
                raise ValueError(f"Unsupported filter operator: {operator}")

        return work

    def aggregate_for_chart( self, df: pd.DataFrame, group_by: list[str],
        metrics: dict[str, str],
    ) -> pd.DataFrame:
        
        self._validate_columns(df, group_by)

        for column, aggregate in metrics.items():
            self._validate_columns(df, [column])

            if aggregate not in self.ALLOWED_AGGS:
                raise ValueError(f"Unsupported aggregate: {aggregate}")

        return (df.groupby(group_by, dropna=False, as_index=False).agg(metrics))

    def sort_rows(
        self,
        df: pd.DataFrame,
        sort_by: str | list[str],
        ascending: bool = True,
    ) -> pd.DataFrame:
        sort_columns = self._as_list(sort_by)
        self._validate_columns(df, sort_columns)

        return df.sort_values(
            sort_columns,
            ascending=ascending,
        )

    def limit_rows(
        self,
        df: pd.DataFrame,
        n: int,
    ) -> pd.DataFrame:
        return df.head(n)

    def select_columns(
        self,
        df: pd.DataFrame,
        columns: list[str],
    ) -> pd.DataFrame:
        self._validate_columns(df, columns)
        return df[columns].copy()

    def create_combined_category(
        self,
        df: pd.DataFrame,
        col1: str,
        col2: str,
        new_col: str | None = None,
        sep: str = " / ",
    ) -> pd.DataFrame:
        out = df.copy()

        self._validate_columns(out, [col1, col2])

        new_col = new_col or f"{col1}_{col2}"

        out[new_col] = (
            out[col1].fillna("").astype(str)
            + sep
            + out[col2].fillna("").astype(str)
        )

        return out

    def create_date_bucket(
        self,
        df: pd.DataFrame,
        date_col: str,
        bucket: str,
        new_col: str | None = None,
    ) -> pd.DataFrame:
        out, generated_col = self._bucket_date(
            df,
            date_col,
            bucket,
        )

        if new_col and new_col != generated_col:
            out = out.rename(
                columns={generated_col: new_col}
            )

        return out

    def select_top_entities(
        self,
        df: pd.DataFrame,
        entity_col: str,
        metric_col: str,
        n: int,
        aggregate: str = "sum",
        ascending: bool = False,
        filters: list[dict] | None = None,
        keep_all_rows: bool = True,
    ) -> pd.DataFrame:
        
        self._validate_columns(df,[entity_col, metric_col])
        if aggregate not in self.ALLOWED_AGGS:
            raise ValueError(f"Unsupported aggregate: {aggregate}")

        if n <= 0:
            raise ValueError("n must be greater than zero")
        
        ranking_data = df.copy()

        if filters:
            ranking_data = self.filter_rows(
                ranking_data,
                filters,
            )

        ranking = (
            ranking_data
            .groupby(entity_col, dropna=False, as_index=False)[metric_col]
            .agg(aggregate)
            .sort_values(metric_col, ascending=ascending)
            .head(n)
        )

        selected_entities = ranking[entity_col].tolist()

        if keep_all_rows:
            return df[df[entity_col].isin(selected_entities)].copy()

        return ranking



    def run_plot_tool(
        self,
        tool_name: str,
        df: pd.DataFrame,
        args: dict[str, Any],
    ) -> dict:
        plotting_tools = {
            "plot_bar_chart": self.plot_bar_chart,
            "plot_line_chart": self.plot_line_chart,
            "plot_pie_chart": self.plot_pie_chart,
            "plot_scatter_chart": self.plot_scatter_chart,
            "plot_list": self.plot_list,
        }

        if tool_name not in plotting_tools:
            raise ValueError(f"Unknown plot tool: {tool_name}")

        args = self._filter_args(tool_name, args)
        return plotting_tools[tool_name](df=df, **args)

    def format_label(self, name: str) -> str:
        """
        Convert column-like names to display labels.

        Examples:
        - year_quarter -> Year quarter
        - percentage_contribution -> Percentage contribution
        - TOTAL_WATER_INJECTION_VOLUME -> Total water injection volume
        """
        if name is None:
            return ""

        text = str(name).replace("_", " ").strip().lower()
        return text[:1].upper() + text[1:]

    def _as_list(self, value):
        if value is None:
            return []
        return [value] if isinstance(value, str) else list(value)

    def _strip_markdown_json(self, text: str) -> str:
        """
        Remove markdown code fences from LLM JSON responses.

        Examples:
        ```json
        {...}
        ```

        ->
        {...}
        """

        text = text.strip()

        text = re.sub(r"^```(?:json)?\s*", "", text)
        text = re.sub(r"\s*```$", "", text)

        return text.strip()

    def _validate_columns(
        self,
        df: pd.DataFrame,
        columns: list[str],
        label: str = "column",
    ):
        """
        Validate that all requested columns exist in the dataframe.
        """

        missing = [c for c in columns if c not in df.columns]

        if missing:
            raise ValueError(f"Missing {label}(s): {missing}")

    def _filter_args(
        self,
        tool_name: str,
        args: dict[str, Any],
    ) -> dict[str, Any]:
        """
        Remove unsupported arguments generated by the LLM.
        """

        allowed_args = {
            "plot_bar_chart": {
                "x",
                "y",
                "aggregate",
                "group_by",
                "color_by",
                "orientation",
                "barmode",
                "title",
                "template",
            },
            "plot_line_chart": {
                "x",
                "y",
                "aggregate",
                "group_by","series_by",
                "color_by",
                "date_bucket",
                "cumulative",
                "title",
                "template",
            },
            "plot_pie_chart": {
                "labels",
                "values",
                "aggregate",
                "group_by",
                "title",
                "hole",
                "template",
            },
            "plot_scatter_chart": {
                "x",
                "y",
                "color_by",
                "size_by",
                "text_by",
                "title",
                "template",
            },
            "plot_list": {
                "columns",
                "sort_by",
                "sort_order",
                "limit",
                "title",
            },
        }

        if tool_name not in allowed_args:
            raise ValueError(f"Unknown tool: {tool_name}")

        return {
            k: v
            for k, v in args.items()
            if k in allowed_args[tool_name]
        }

    def _aggregate(
        self,
        df: pd.DataFrame,
        group_by: list[str],
        value_cols: list[str],
        aggregate: str,
    ) -> pd.DataFrame:
        if aggregate not in self.ALLOWED_AGGS:
            raise ValueError(f"Unsupported aggregate: {aggregate}")

        self._validate_columns(df, group_by, "group_by column")
        self._validate_columns(df, value_cols, "value column")

        return (
            df.groupby(group_by, dropna=False, as_index=False)[value_cols]
            .agg(aggregate)
        )

    def _bucket_date(
        self,
        df: pd.DataFrame,
        date_col: str,
        bucket: str,
    ) -> tuple[pd.DataFrame, str]:
        """
        Create a date bucket column.

        bucket:
        - "D": day
        - "W": week
        - "M": month
        - "Q": quarter
        - "Y": year
        """

        out = df.copy()
        bucket_col = f"{date_col}_{bucket}"

        self._validate_columns(out, [date_col])

        out[date_col] = pd.to_datetime(out[date_col], errors="coerce")

        if bucket == "D":
            out[bucket_col] = out[date_col].dt.to_period("D").dt.to_timestamp()
        elif bucket == "W":
            out[bucket_col] = out[date_col].dt.to_period("W").dt.start_time
        elif bucket == "M":
            out[bucket_col] = out[date_col].dt.to_period("M").dt.to_timestamp()
        elif bucket == "Q":
            out[bucket_col] = out[date_col].dt.to_period("Q").dt.to_timestamp()
        elif bucket == "Y":
            out[bucket_col] = out[date_col].dt.to_period("Y").dt.to_timestamp()
        else:
            raise ValueError("date_bucket must be one of: D, W, M, Q, Y")

        return out, bucket_col

    def plot_bar_chart(
        self,
        df: pd.DataFrame,
        x: str,
        y: str | list[str],
        *,
        aggregate: str | None = None,
        group_by: list[str] | str | None = None,
        color_by: str | None = None,
        orientation: str = "v",
        barmode: str = "group",
        title: str | None = None,
        # template: str = "plotly_white",
    ) -> dict:
        y_cols = self._as_list(y)

        required = [x, *y_cols]
        if color_by:
            required.append(color_by)

        self._validate_columns(df, required)

        work = df.copy()

        if aggregate is not None:
            group_cols = self._as_list(group_by) or [x]

            if x not in group_cols:
                group_cols.insert(0, x)

            if color_by and color_by not in group_cols:
                group_cols.append(color_by)

            work = self._aggregate(work, group_cols, y_cols, aggregate)

        data = []
        groups = work.groupby(color_by, dropna=False) if color_by else [(None, work)]

        for group_value, g in groups:
            for y_col in y_cols:
                if group_value is None:
                    name = self.format_label(y_col)
                else:
                    name = self.format_label(str(group_value))

                if group_value is not None and len(y_cols) > 1:
                    name = f"{self.format_label(str(group_value))} - {self.format_label(y_col)}"

                trace = {
                    "type": "bar",
                    "name": name,
                    "orientation": orientation,
                }

                if orientation == "h":
                    trace["x"] = g[y_col].tolist()
                    trace["y"] = g[x].astype(str).tolist()
                else:
                    trace["x"] = g[x].astype(str).tolist()
                    trace["y"] = g[y_col].tolist()

                data.append(trace)

        y_label = self.format_label(", ".join(y_cols))
        x_label = self.format_label(x)

        return {
            "data": data,
            "layout": {
                "title": {
                    "text": title or f"{y_label} by {x_label}"
                },
                "xaxis": {
                    "title": {
                        "text": y_label if orientation == "h" else x_label
                    }
                },
                "yaxis": {
                    "title": {
                        "text": x_label if orientation == "h" else y_label
                    }
                },
                "barmode": barmode,
                # "template": template,
            },
            "config": {
                "responsive": True,
                "displaylogo": False,
            },
        }

    def plot_line_chart(
        self,
        df: pd.DataFrame,
        x: str,
        y: str | list[str],
        *,
        aggregate: str | None = None,
        group_by: list[str] | str | None = None,
        series_by: str | None = None,
        color_by: str | None = None,  # backwards compatibility
        date_bucket: str | None = None,
        cumulative: bool = False,
        title: str | None = None,
        # template: str = "plotly_white",
    ) -> dict:
        y_cols = self._as_list(y)

        if series_by is None:
            series_by = color_by

        required = [x, *y_cols]
        if series_by:
            required.append(series_by)

        self._validate_columns(df, required)

        work = df.copy()
        x_plot = x

        if date_bucket is not None:
            work, x_plot = self._bucket_date(work, x, date_bucket)

        if aggregate is not None:
            group_cols = self._as_list(group_by) or [x_plot]

            if x_plot not in group_cols:
                group_cols.insert(0, x_plot)

            if series_by and series_by not in group_cols:
                group_cols.append(series_by)

            work = self._aggregate(work, group_cols, y_cols, aggregate)

        sort_cols = [series_by, x_plot] if series_by else [x_plot]
        work = work.sort_values(sort_cols)

        if cumulative:
            if series_by:
                for col in y_cols:
                    work[col] = work.groupby(series_by, dropna=False)[col].cumsum()
            else:
                for col in y_cols:
                    work[col] = work[col].cumsum()

        data = []
        groups = work.groupby(series_by, dropna=False) if series_by else [(None, work)]

        total_points = len(work) * len(y_cols)
        disable_all_markers = total_points > 2000

        for group_value, g in groups:
            for y_col in y_cols:
                if group_value is None:
                    name = self.format_label(y_col)
                elif len(y_cols) == 1:
                    name = str(group_value)
                else:
                    name = f"{group_value} - {self.format_label(y_col)}"

                data.append({
                    "type": "scatter",
                    "mode": "lines",
                    "x": g[x_plot].tolist(),
                    "y": g[y_col].tolist(),
                    "name": name,
                })

        return {
            "data": data,
            "layout": {
                "title": {
                    "text": self.format_label(title) or f"{', '.join(y_cols)} over {x}"
                },
                "xaxis": {
                    "title": {
                        "text": self.format_label(x)
                    }
                },
                "yaxis": {
                    "title": {
                        "text": self.format_label(", ".join(y_cols))
                    }
                },
            },
            "config": self.plotly_config,
        }

    def plot_pie_chart(
        self,
        df: pd.DataFrame,
        labels: str,
        values: str,
        *,
        aggregate: str | None = None,
        group_by: list[str] | str | None = None,
        title: str | None = None,
        hole: float = 0.0,
        # template: str = "plotly_white",
    ) -> dict:
        self._validate_columns(df, [labels, values])

        work = df.copy()

        if aggregate is not None:
            group_cols = self._as_list(group_by) or [labels]

            if labels not in group_cols:
                group_cols.insert(0, labels)

            work = self._aggregate(work, group_cols, [values], aggregate)

        return {
            "data": [
                {
                    "type": "pie",
                    "labels": work[labels].astype(str).tolist(),
                    "values": work[values].tolist(),
                    "hole": hole,
                }
            ],
            "layout": {
                "title": {
                    "text": title or f"{values} share by {labels}"
                },
                # "template": template,
            },
            "config": self.plotly_config
        }

    def plot_scatter_chart(
        self,
        df: pd.DataFrame,
        x: str,
        y: str | list[str],
        *,
        color_by: str | None = None,
        size_by: str | None = None,
        text_by: str | None = None,
        title: str | None = None,
        # template: str = "plotly_white",
    ) -> dict:
        y_cols = self._as_list(y)

        required = [x, *y_cols]
        if color_by:
            required.append(color_by)
        if size_by:
            required.append(size_by)
        if text_by:
            required.append(text_by)

        self._validate_columns(df, required)

        work = df.copy()
        data = []
        groups = work.groupby(color_by, dropna=False) if color_by else [(None, work)]

        total_points = len(work) * len(y_cols)
        disable_all_markers = total_points > 2000

        for group_value, g in groups:
            for y_col in y_cols:
                name = y_col if group_value is None else str(group_value)

                if group_value is not None and len(y_cols) > 1:
                    name = f"{group_value} - {y_col}"

                n_points = len(g)

                use_markers = (
                    not disable_all_markers
                    and n_points <= 100
                )

                mode = "markers" if use_markers else "lines"

                trace = {
                    "type": "scattergl",
                    "mode": mode,
                    "x": g[x].tolist(),
                    "y": g[y_col].tolist(),
                    "name": name,
                }

                if size_by:
                    size_values = pd.to_numeric(g[size_by], errors="coerce").fillna(0)
                    max_size = max(float(size_values.max()), 1.0)

                    trace["marker"] = {
                        "size": size_values.tolist(),
                        "sizemode": "area",
                        "sizeref": max_size / 40,
                        "sizemin": 4,
                    }

                if text_by:
                    trace["text"] = g[text_by].astype(str).tolist()
                    trace["hovertemplate"] = (
                        f"{x}: %{{x}}<br>"
                        f"{y_col}: %{{y}}<br>"
                        f"{text_by}: %{{text}}"
                        "<extra></extra>"
                    )

                data.append(trace)

        return {
            "data": data,
            "layout": {
                "title": {
                    "text": title or f"{', '.join(y_cols)} vs {x}"
                },
                "xaxis": {
                    "title": {
                        "text": x
                    }
                },
                "yaxis": {
                    "title": {
                        "text": ", ".join(y_cols)
                    }
                },
                # "template": template,
            },
            "config": self.plotly_config
        }

    def plot_list(
        self,
        df: pd.DataFrame,
        columns: list[str] | str | None = None,
        *,
        sort_by: str | None = None,
        sort_order: str = "desc",
        limit: int | None = None,
        title: str | None = None,
    ) -> dict:
        work = df.copy()

        if columns is not None:
            columns = self._as_list(columns)
            self._validate_columns(work, columns)
            work = work[columns]

        if sort_by is not None:
            self._validate_columns(work, [sort_by])

            ascending = sort_order.lower() == "asc"
            work = work.sort_values(sort_by, ascending=ascending)

        if limit is not None:
            work = work.head(limit)

        header_values = [self.format_label(c) for c in work.columns]

        cell_values = []
        for col in work.columns:
            s = work[col]

            if pd.api.types.is_datetime64_any_dtype(s):
                values = s.dt.strftime("%Y-%m-%d").fillna("").tolist()
            else:
                values = s.fillna("").astype(str).tolist()

            cell_values.append(values)

        return {
            "data": [
                {
                    "type": "table",
                    "header": {
                        "values": header_values,
                        "align": "left",
                    },
                    "cells": {
                        "values": cell_values,
                        "align": "left",
                    },
                }
            ],
            "layout": {
                "title": {
                    "text": self.format_label(title) or "Table"
                },
            },
            "config": self.plotly_config
        }
    

class PresenterConfig:

    prompt :str = chart_agent_prompt
    split_subinstructions_prompr: str = split_subinstructions_prompt 
    small_table_prompt: str = small_table_prompt 
    
    def __init__(
        self,
        charting_tools: PresenterChartingTools | None = None,
    ):
        self.charting_tools = charting_tools or PresenterChartingTools()


class PresenterComponent4:
    """
    Converts an ExecutorState into UI display items.

    Each TaskResult is processed as a whole:
    - all TextResult and DataFrameResult objects are added to one context;
    - one LLM call splits the task instruction into sub-instructions;
    - each sub-instruction is associated with one source;
    - text sources become text UIItems;
    - table sources are passed to the existing dataframe presentation logic.
    """

    def __init__(
        self,
        llm: Any,
        config: PresenterConfig | None = None,
    ):
        self.llm = llm
        self.config = config or PresenterConfig()
        self.charting_tools = self.config.charting_tools

    def run(self, result_state: ExecutorState) -> PresenterResponse:
        return self.process_task_results(result_state)

    def _make_clarification_item(
        self,
        clarification_request: str,
    ) -> UIItem:
        return UIItem(
            id=f"question_{uuid4().hex[:8]}",
            type="question",
            title="Additional information required",
            data={"question": clarification_request},
        )

    def _make_text_item(
        self,
        data_result: TextResult,
    ) -> UIItem:
        return UIItem(
            id=f"text_{uuid4().hex[:8]}",
            type="text",
            title=None,
            data={"text": data_result.text},
        )

    def _make_error_item(
        self,
        task_result: TaskResult,
        data_result: object | None = None,
    ) -> UIItem:
        return UIItem(
            id=f"error_{uuid4().hex[:8]}",
            type="error",
            title="Presentation error",
            data={
                "message": f"No presenter for result from {task_result.agent}",
                "details": (
                    str(type(data_result))
                    if data_result is not None
                    else task_result.instruction
                ),
            },
        )

    def build_task_result_context(
        self,
        task_result: TaskResult,
    ) -> tuple[str, dict[str, TextResult | DataFrameResult]]:
        """
        Build:
        - one text context containing all TextResult and DataFrameResult objects;
        - a source map used later to recover the original result objects.
        """
        processor = TableResponseProcessor()

        context_parts: list[str] = []
        source_map: dict[str, TextResult | DataFrameResult] = {}

        for n, data_result in enumerate(task_result.data_results):

            if isinstance(data_result, TextResult):
                source_id = f"text_{n}"

                context_parts.append(
                    "\n".join([
                        f"SOURCE_ID: {source_id}",
                        "SOURCE_TYPE: text",
                        "CONTENT:",
                        data_result.text,
                    ])
                )

                source_map[source_id] = data_result

            elif isinstance(data_result, DataFrameResult):
                source_id = data_result.table_name
                table_context = processor.extract_table_context(data_result)

                context_parts.append(
                    "\n".join([
                        f"SOURCE_ID: {source_id}",
                        "SOURCE_TYPE: table",
                        table_context,
                    ])
                )

                source_map[source_id] = data_result

        context_text = "\n\n---\n\n".join(context_parts)

        return context_text, source_map

    def get_subinstructions(
        self,
        task_result: TaskResult,
        context_text: str,
    ) -> SubInstructions:
        """
        Split the task instruction and associate each sub-instruction
        with one available source.
        """
        messages = [
            SystemMessage(content=self.config.split_subinstructions_prompr),
            HumanMessage(
                content=(
                    f"INSTRUCTION\n"
                    f"{task_result.instruction}\n\n"
                    f"AVAILABLE SOURCES\n"
                    f"{context_text}"
                )
            ),
        ]

        structured_llm = self.llm.with_structured_output(SubInstructions)

        return structured_llm.invoke(messages)

    def process_single_task_result(
        self,
        task_result: TaskResult,
    ) -> list[UIItem]:
        context_text, source_map = self.build_task_result_context(
            task_result
        )

        sub_instructions = self.get_subinstructions(
            task_result=task_result,
            context_text=context_text,
        )

        ui_items: list[UIItem] = []

        for item in sub_instructions.items:
            source = source_map[item.source_id]

            if item.kind == "text":
                ui_items.append(
                    self._make_text_item(source)
                )

            elif item.kind == "table":
                ui_items.append(
                    self._process_dataframe(
                        source,
                        item.sub_instruction,
                    )
                )

        return ui_items

    def process_task_results(
        self,
        execution_state: ExecutorState,
    ) -> PresenterResponse:
        ui_items: list[UIItem] = []

        clarification_request = execution_state.get(
            "clarification_request"
        )

        if clarification_request:
            ui_items.append(
                self._make_clarification_item(
                    clarification_request
                )
            )

            return PresenterResponse(items=ui_items)

        for task_result in execution_state.get("task_results", []):
            ui_task_items = self.process_single_task_result(
                task_result
            )

            ui_items.extend(ui_task_items)

        return PresenterResponse(items=ui_items)

    def _present_very_small_table(
        self,
        df: pd.DataFrame,
        data_result: DataFrameResult,
        instruction: str,
    ) -> UIItem:
        data_string = df.to_json()

        print('processing very small table')
        prompt = (
            self.config.small_table_prompt
            + "\n\n"
            + (
                "### Context\n"
                f"- Table Name: {getattr(data_result, 'table_name', 'N/A')}\n"
                f"- Description: "
                f"{getattr(data_result, 'description', 'No description provided.')}\n\n"
                "### Data\n"
                f"{data_string}\n\n"
                "User question:\n"
                f"{instruction}\n"
            )
        )

        response = self.llm.invoke(prompt)

        text_output = (
            response.content
            if hasattr(response, "content")
            else str(response)
        )

        return UIItem(
            id=f"text_{uuid4().hex[:8]}",
            type="text",
            title=None,
            data={"text": text_output},
        )

    def _make_chart_item(
        self,
        figure_title: str,
        plotly_json_figure: dict,
        description: str | None = None,
    ) -> UIItem:
        return UIItem(
            id=f"chart_{uuid4().hex[:8]}",
            type="chart",
            title=figure_title,
            data={
                "engine": "plotly",
                "plotly": plotly_json_figure,
            },
            meta={
                "description": description,
            },
        )


    def _run_chart_plan(
        self,
        plan: dict,
        df: pd.DataFrame,
    ) -> dict | None:
        work = df.copy()

        preprocess_steps = plan.get("preprocess") or []
        plot = plan.get("plot")

        if preprocess_steps:
            work = self.charting_tools.run_preprocess(
                work,
                preprocess_steps,
            )

        if not plot:
            return None

        tool_name = plot.get("tool")
        args = plot.get("args") or {}

        return self.charting_tools.run_plot_tool(
            tool_name=tool_name,
            df=work,
            args=args,
        )


    def _format_label(
        self,
        name: str,
    ) -> str:
        if name is None:
            return ""

        text = str(name).replace("_", " ").strip().lower()

        return text[:1].upper() + text[1:]

    def _process_dataframe(
        self,
        data_result: DataFrameResult,
        instruction: str,
    ) -> UIItem:
        df = data_result.dataframe
        nrows, ncols = df.shape

        if nrows <= 2 and ncols <= 2:
            return self._present_very_small_table(
                df,
                data_result,
                instruction,
            )

        processor = TableResponseProcessor()

        table_context = processor.extract_table_context(
            data_result
        )

        chart_plan = self._select_chart_plan(
            instruction,
            table_context,
        )

        print('****chart plan****')
        print(instruction)
        print(chart_plan)


        chart_output = self._run_chart_plan(
            chart_plan,
            df,
        )

        return self._make_chart_item(
            self._format_label(data_result.table_name),
            chart_output,
            data_result.description,
        )

    def _strip_markdown_json(
        self,
        text: str,
    ) -> str:
        text = text.strip()

        text = re.sub(
            r"^```(?:json)?\s*",
            "",
            text,
        )

        text = re.sub(
            r"\s*```$",
            "",
            text,
        )

        return text.strip()

    def _select_chart_plan(
        self,
        user_query: str,
        table_context: str,
    ) -> dict:
        messages = [
            SystemMessage(content=self.config.prompt),
            HumanMessage(
                content=(
                    f"USER QUERY\n"
                    f"{user_query}\n\n"
                    f"TABLE\n"
                    f"{table_context}"
                )
            ),
        ]

        response = self.llm.invoke(messages)

        text = self._strip_markdown_json(
            response.content
        )

        return json.loads(text)
    

    

zero temp, seed 42, top_p = 1
****chart plan****
Identify the top 5 producer wells based on their cumulative oil production in 2018.
{'reason': 'The table already contains the top 5 producer wells based on their cumulative oil production in 2018, so no preprocessing is needed.', 'preprocess': [], 'plot': {'tool': 'plot_bar_chart', 'args': {'x': 'NAME', 'y': 'total_oil_volume', 'aggregate': None, 'group_by': ['NAME'], 'color_by': None, 'orientation': 'v', 'barmode': 'group', 'title': 'Top 5 Producer Wells by Cumulative Oil Production in 2018'}}}
****chart plan****
Plot the yearly liquid production for the top 5 producer wells.
{'reason': 'The table already contains yearly liquid production data for the top 5 producer wells, so no preprocessing is needed.', 'preprocess': [], 'plot': {'tool': 'plot_line_chart', 'args': {'x': 'YEAR', 'y': 'total_liquid_volume', 'aggregate': None, 'group_by': ['YEAR', 'NAME'], 'series_by': 'NAME', 'date_bucket': None, 'cumulative': False, 'title': 'Yearly L

[UIItem(id='text_6d571369', type='text', title=None, data={'text': 'Voidage Replacement Ratio (VRR) is a key reservoir management metric that measures the ratio of the volume of injected fluids (e.g., water, gas) to the volume of produced reservoir fluids (oil, gas, and water). It is used to assess whether the voidage created by production is being adequately replaced to maintain reservoir pressure and optimize recovery. \n\nA VRR of 1.0 indicates full replacement, helping to sustain pressure and improve sweep efficiency, while a VRR less than 1.0 suggests under-replacement, potentially leading to pressure decline. Conversely, a VRR greater than 1.0 may indicate over-injection, which could lead to operational inefficiencies or formation damage.'}, meta={}),
 UIItem(id='chart_c408deeb', type='chart', title='Top 5 producers 2018', data={'engine': 'plotly', 'plotly': {'data': [{'type': 'bar', 'name': 'Total oil volume', 'orientation': 'v', 'x': ['P4', 'P3', 'P1', 'P2'], 'y': [12875.199035

Voidage Replacement Ratio (VRR) is a key reservoir management metric that measures the ratio of the volume of injected fluids (e.g., water, gas) to the volume of produced reservoir fluids (oil, gas, and water). It is used to assess whether the voidage created by production is being adequately replaced to maintain reservoir pressure and optimize recovery. 

A VRR of 1.0 indicates full replacement, helping to sustain pressure and improve sweep efficiency, while a VRR less than 1.0 suggests under-replacement, potentially leading to pressure decline. Conversely, a VRR greater than 1.0 may indicate over-injection, which could lead to operational inefficiencies or formation damage.


In [ ]:
#pprint.pprint( execution_state['task_results'][1] )
t = execution_state['task_results'][1]
instruction = t.instruction 
#t.data_results.insert(0, TextResult(text="All the fruits in the basket are sweet."))

print( instruction )
p = TableResponseProcessor()
context = [] 

aux= {} 
for n,data_result in enumerate(t.data_results):


    if isinstance( data_result, TextResult):
        print("processing text ")
        context.append( data_result.text )
        aux[n] = data_result.text 

    if isinstance(data_result, DataFrameResult):
        print("processing dataframe result")
        table_context = p.extract_table_context( data_result )
        context.append( table_context )
        aux[ data_result.table_name ] = (table_context,data_result)
    
prompt = """You will receive an 'instruction' and information of tables and text. your job is to 
analyze the instruction. It might contain several sub-instructions. 
Decide what parts of the information available can be used to execute the 
instruction and its sub-instructions. 

Do not explain anything, do not add more details than strictly needed to produce the required output 
"""
class SubInstruction(BaseModel):
    sub_instruction: str = Field(
        description="The specific part of the instruction."
    )
    kind:  Literal['table','text']
    
    data: str  = Field(
        description="The name of the table or the textual information "
    )
  
  
class SubInstructions(BaseModel):

    items: List[SubInstruction] =  Field(description="The list of sub-instructions and the information relevant for each")

context = "\n\n".join(context)

#instruction2 = "Check if the fruits are sour or sweet, " + instruction# then list the top 5 producers by cummulated oil produced in 2018 "

context =  "\n" + context + "\n\n" + instruction + "\n\n"
messages = [SystemMessage(prompt + "\n\n" + context )]

structured_llm = llm.with_structured_output(SubInstructions)
response = structured_llm.invoke(messages)

In [ ]:
pprint.pprint(context)


In [ ]:
pprint.pprint(response.items)


In [ ]:

items = [] 

for i in response.items:
    if i.kind=='text':
        print("It is a text")
        r =  UIItem(
                id=f"text_{uuid4().hex[:8]}",
                type="text",
                title=None,
                data={"text": i.data}
            )
        items.append( r )

    if i.kind=='table':

        #print( '**',i.sub_instruction,'**')
        table_name = i.data 
        acontext = aux[table_name ][0]
        data_result = aux[table_name][1]
        #print('table', table_name, acontext )
        
        r = presenter._process_dataframe(data_result,i.sub_instruction)
        items.append( r )

        


In [ ]:
print( items )

print()
print()
print()

for item in items:

    if item.type=='text':
        print( item.data['text'])

    item = item.data.get('plotly',None)
    if item:
        pio.show(item)


## More organized

In [ ]:
from typing import Literal
from uuid import uuid4

from pydantic import BaseModel, Field
from langchain_core.messages import SystemMessage, HumanMessage


# =============================================================================
# Structured output models
# =============================================================================

class SubInstruction(BaseModel):
    sub_instruction: str = Field(
        description=(
            "The specific part of the original instruction that must be "
            "presented using the selected source."
        )
    )

    kind: Literal["table", "text"] = Field(
        description="The type of source associated with this sub-instruction."
    )

    source_id: str = Field(
        description=(
            "The exact SOURCE ID provided in the available sources. "
            "For a table, this is the exact table name. "
            "For text, this is the exact text result identifier."
        )
    )


class SubInstructions(BaseModel):
    items: list[SubInstruction] = Field(
        description=(
            "The requested outputs, in the order in which they should be "
            "presented. Irrelevant and intermediate sources must be omitted."
        )
    )


# =============================================================================
# Routing prompt
# =============================================================================

TASK_PRESENTATION_ROUTER_PROMPT = """
You receive one instruction and a set of available sources.

The instruction may contain several sub-instructions.

Your job is to:

1. Identify each distinct output explicitly requested by the instruction.
2. Match each requested output to exactly one relevant source.
3. Return the outputs in the order in which they should be presented.
4. Use the exact SOURCE ID provided for each source.
5. Omit sources that are irrelevant or only intermediate calculation results.
6. Do not invent facts, tables, source IDs, calculations, or additional requests.
7. Do not explain your decisions.
8. Do not create an item when the available sources cannot support it.
9. Do not repeat the same source unless it is genuinely required for two
   different requested outputs.

For a text source:
- Use it when the source directly contains the requested textual answer.
- The sub_instruction should describe the part of the instruction answered
  by the text.
- The source_id must be the exact text SOURCE ID.

For a table source:
- Use it when the table contains the information required for the requested
  table, chart, list, ranking, comparison, or numerical presentation.
- The sub_instruction must contain only the part of the original instruction
  that the selected table can address.
- The source_id must be the exact table SOURCE ID.

Important:
- A table used only to calculate another final table is usually an intermediate
  source and should be omitted unless the user explicitly requested it.
- Do not return the source content itself.
- Return only the structured result.
"""


# =============================================================================
# Select the TaskResult to process
# =============================================================================

task_result_index = 1

task_result = execution_state["task_results"][task_result_index]
instruction = task_result.instruction

print("TASK INSTRUCTION")
print(instruction)
print()


# =============================================================================
# Build source context and source lookup
# =============================================================================

table_processor = TableResponseProcessor()

source_contexts: list[str] = []
source_lookup: dict[str, TextResult | DataFrameResult] = {}

for result_index, data_result in enumerate(task_result.data_results):

    if isinstance(data_result, TextResult):
        source_id = f"text_result_{result_index}"

        source_contexts.append(
            "\n".join([
                f"SOURCE ID: {source_id}",
                "SOURCE KIND: text",
                "CONTENT:",
                data_result.text,
            ])
        )

        source_lookup[source_id] = data_result

    elif isinstance(data_result, DataFrameResult):
        source_id = data_result.table_name
        table_context = table_processor.extract_table_context(data_result)

        source_contexts.append(
            "\n".join([
                f"SOURCE ID: {source_id}",
                "SOURCE KIND: table",
                table_context,
            ])
        )

        source_lookup[source_id] = data_result

    else:
        print(
            "Ignoring unsupported data result:",
            type(data_result).__name__,
        )


available_sources_context = "\n\n---\n\n".join(source_contexts)

print("AVAILABLE SOURCE IDS")
for source_id, source in source_lookup.items():
    print(f"- {source_id}: {type(source).__name__}")
print()


# =============================================================================
# One LLM call to split the instruction and route each part to a source
# =============================================================================

messages = [
    SystemMessage(content=TASK_PRESENTATION_ROUTER_PROMPT),
    HumanMessage(
        content=(
            f"ORIGINAL INSTRUCTION\n"
            f"{instruction}\n\n"
            f"AVAILABLE SOURCES\n"
            f"{available_sources_context}"
        )
    ),
]

structured_llm = llm.with_structured_output(SubInstructions)
routing_response = structured_llm.invoke(messages)

print("ROUTING RESPONSE")
for routed_item in routing_response.items:
    print(routed_item)
print()


# =============================================================================
# Convert the routed outputs into UIItems
# =============================================================================

items: list[UIItem] = []

for routed_item in routing_response.items:
    source = source_lookup.get(routed_item.source_id)

    if source is None:
        items.append(
            UIItem(
                id=f"error_{uuid4().hex[:8]}",
                type="error",
                title="Presentation error",
                data={
                    "message": (
                        "The presentation router selected an unknown source."
                    ),
                    "details": routed_item.source_id,
                },
            )
        )
        continue

    if routed_item.kind == "text":
        if not isinstance(source, TextResult):
            items.append(
                UIItem(
                    id=f"error_{uuid4().hex[:8]}",
                    type="error",
                    title="Presentation error",
                    data={
                        "message": (
                            "The presentation router classified a non-text "
                            "source as text."
                        ),
                        "details": routed_item.source_id,
                    },
                )
            )
            continue

        items.append(
            UIItem(
                id=f"text_{uuid4().hex[:8]}",
                type="text",
                title=None,
                data={
                    "text": source.text,
                },
                meta={
                    "sub_instruction": routed_item.sub_instruction,
                    "source_id": routed_item.source_id,
                },
            )
        )

    elif routed_item.kind == "table":
        if not isinstance(source, DataFrameResult):
            items.append(
                UIItem(
                    id=f"error_{uuid4().hex[:8]}",
                    type="error",
                    title="Presentation error",
                    data={
                        "message": (
                            "The presentation router classified a non-table "
                            "source as a table."
                        ),
                        "details": routed_item.source_id,
                    },
                )
            )
            continue

        try:
            ui_item = presenter._process_dataframe(
                source,
                routed_item.sub_instruction,
            )

            if ui_item is not None:
                ui_item.meta = {
                    **ui_item.meta,
                    "sub_instruction": routed_item.sub_instruction,
                    "source_id": routed_item.source_id,
                }
                items.append(ui_item)

        except Exception as exc:
            items.append(
                UIItem(
                    id=f"error_{uuid4().hex[:8]}",
                    type="error",
                    title="Presentation error",
                    data={
                        "message": (
                            f"Could not present table "
                            f"{routed_item.source_id}."
                        ),
                        "details": str(exc),
                    },
                    meta={
                        "sub_instruction": routed_item.sub_instruction,
                        "source_id": routed_item.source_id,
                    },
                )
            )


# =============================================================================
# Final presenter response
# =============================================================================

presenter_response = PresenterResponse(items=items)

print("GENERATED UI ITEMS")
for item in presenter_response.items:
    print(
        {
            "id": item.id,
            "type": item.type,
            "title": item.title,
            "source_id": item.meta.get("source_id"),
            "sub_instruction": item.meta.get("sub_instruction"),
        }
    )

presenter_response
  

In [ ]:
items = presenter_response.items 
print( items )

print()
print()
print()

for item in items:

    if item.type=='text':
        print( item.data['text'])

    item = item.data.get('plotly',None)
    if item:
        pio.show(item)


In [ ]:


presenter = PresenterComponent4( llm )
#presenter = PresenterComponent2( llm )

ui_items = presenter.run( execution_state )
ui_items

In [ ]:
ui_items.items

In [ ]:
item = ui_items.items[1]
item = item.data['plotly']
pio.show(item)

item = ui_items.items[2]
item = item.data['plotly']
pio.show(item)

item = ui_items.items[3]
item = item.data['plotly']
pio.show(item)